Using ML techniques to predict RT side effect as patients who affected by cancer and took RT after removing cancer part could be affercted by some RT side effect which can influence their quality of life such as arm_lymphoedema side effect. The data that had been collected contain clinical and genetic data  

In [ ]:
##### 'breast_Arm_Lymphoedema24m' ###########

**1. Strategy: matching the problem with the solution**

**Arm Lymphedema**

* Key Time Points: 3 months, 12 months, 24 months, and 36 months.
* Why: Lymphedema often begins to develop within the first year post-treatment, with swelling and fluid accumulation in the arm. The condition may worsen or stabilize in the subsequent years, but typically shows early signs that can persist or fluctuate over time.

**2. Dataset preparation and preprocessing**

**2.1) Data collection**



The data is stores as csv file and previously had been taken from the hospitals of 9 regiens, and all of these pateints had breast cancer before and acure with RT afterward the operation.

**2.1.1) importing some libraries in order to upload data**

In [ ]:
# importing Pandas and NumPy library
import pandas as pd
import numpy as np
from numpy import sqrt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# for visualising and metrics
from sklearn.decomposition import PCA
from sklearn import metrics
# for color heatmap
from matplotlib import cm

# for error
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from numpy import sqrt, argmax

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Step 1: Dynamically compute class weights
from collections import Counter
import numpy as np

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
import matplotlib.cm as cm

## This with pipeline of standarise scaling

from tqdm import tqdm

**2.1.2) uploading the data and safe them with differnt names**

In [ ]:
# Read the data from original file (use relative paths)
data1 = pd.read_csv('/content/FinalRTmerged_data28.12.24.csv')
#data1 = pd.read_csv('/content/FinalRTmerged_data8.10.25.csv')
data2= pd.read_csv('/content/ArmlymphedemaSNPs.csv')


In [ ]:
list(data1.columns)

In [ ]:
#data1.columns = data1.columns.str.replace(' ', '_')

In [ ]:
# Select the dependent variable here
dependent1 = 'breast_arm_lymphodema_60m'


**2.1.3) implement some functions**

In [ ]:
# To make the text bold
def bold(text):
  return "\033[1m" + text + "\033[0m"

In [ ]:
# To chech the percenage of null value in each feature
def nullP(df):
  print(((pd.isnull(df).sum()/len(df))*100).round())

In [ ]:
# calculate uniq data
def Cuniq(df, x):
  print(df[x].sort_values().value_counts())

In [ ]:
# This finction is used to print the uniques values in each feature in the dataset
def uniq(data):
  for col in list(data):
    print(col, "Unique Values are:\n ", data[col].unique())
    print("===================================================================================")

In [ ]:
# This function is used to Remove all features that contain X % Missing values
def X_per_removed(data1, perc):
  #Remove all features that contain 10% or more null
  min_count = int(((100 - perc) / 100) * data1.shape[0] + 1)
  feat_data = data1.dropna(axis=1, thresh=min_count)  # Drop columns which contain more half than of their values are missing
  print(feat_data.shape)
  print(list(feat_data.columns))  # then added the selected features to the main feature list


In [ ]:
# to show all rows and columns in dataframe
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [ ]:
def pairwisecorr(df, col, threshold):
    if col:
      X = df.loc[:, df.columns != col]
    else:
      X = df.copy()

    # Exclude datetime columns and string columns from variance calculation
    numeric_features = X.select_dtypes(include=np.number).columns
    X_numeric = X[numeric_features]

    # Select only numeric features for correlation calculation
    X_for_corr = X[numeric_features] # This line is added

    corr_matrix = X_for_corr.corr().abs()  # Compute the correlation matrix and take absolute values

    # Create a True/False mask to ignore the upper triangle (since correlation is symmetric)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    tri_df = corr_matrix.mask(mask)

    # Initialize a list to store features to drop and a dictionary to store correlated pairs
    to_drop = []
    correlated_pairs = []

    # Find all pairs of features that have a correlation above the threshold
    for c in tri_df.columns:
        for index, value in tri_df[c].items():
            if value > threshold:
                correlated_pairs.append((c, index, value))
                # Decide which feature to drop based on variance, only for numeric features
                if c in numeric_features and index in numeric_features:
                    if X_numeric[c].var() > X_numeric[index].var():
                        to_drop.append(index)
                    else:
                        to_drop.append(c)
                else:
                    # Handle non-numeric features (e.g., drop one arbitrarily)
                    to_drop.append(index)  # Or to_drop.append(c)

    # Drop duplicate features from the list
    to_drop = list(set(to_drop))

    # Drop the highly correlated features
    reduced_X = X.drop(to_drop, axis=1)

    # Display the correlated pairs and their correlation values
    print("Highly correlated feature pairs (correlation > {:.2f}):".format(threshold))
    for pair in correlated_pairs:
        print(f"Features: {pair[0]} and {pair[1]}, Correlation: {pair[2]:.2f}")

    # Print the features kept and removed
    print("\nShape of original dataset:", X.shape)
    print("\nRemaining features after dropping highly correlated ones:")
    print(list(reduced_X.columns))
    print("\nThese features were removed due to high correlation:")
    print(to_drop)
    print("\nShape of the reduced dataset:", reduced_X.shape)
    return reduced_X

In [ ]:
# VIF measures how much a feature (column) is correlated with the other features in your dataset. It's used to detect multicollinearity, which is when features are too similar to each other.
#A high VIF (e.g., > 5 or 10) means the feature is highly correlated with other features.
#This can make models like regression unstable or hard to interpret.
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df, threshold):
    # Store Subject_Id separately
    subject_ids = df['Subject_Id']

    # Drop Subject_Id and non-numeric, drop rows with missing values
    df_snp = df.drop(columns=['Subject_Id'])
    df_snp = df_snp.select_dtypes(include=[float, int])
    df_snp = df_snp.dropna()

    # Keep matching Subject_Id values (after dropna)
    subject_ids = subject_ids.loc[df_snp.index]

    dropped = True
    while dropped:
        vif_data = pd.DataFrame()
        vif_data["SNP"] = df_snp.columns
        vif_data["VIF"] = [variance_inflation_factor(df_snp.values, i)
                           for i in range(df_snp.shape[1])]

        max_vif = vif_data["VIF"].max()
        if max_vif > threshold:
            drop_snp = vif_data.sort_values("VIF", ascending=False).iloc[0]["SNP"]
            df_snp = df_snp.drop(columns=[drop_snp])
            print(f"Dropped {drop_snp} with VIF {max_vif:.2f}")
        else:
            dropped = False

    # Add Subject_Id back safely
    df_snp['Subject_Id'] = subject_ids.values

    return df_snp


**2.2) Data visualization**

Which involves presenting data in a visual format such as charts and graphs to make it easier to understand and to identify patterns anf drends in the data.

**2.2.1) Labeling**

In [ ]:
print(data1.shape + data2.shape)

In [ ]:
data1.head()

In [ ]:
print(list(data1.columns))

In [ ]:
data2.head()

In [ ]:
data2.head(2000)

In [ ]:
# Show the length of data
len(data2)

In [ ]:
data1.describe()

In [ ]:
data2.describe()

**2.2.2) Data selection**

In [ ]:
data1.columns = data1.columns.str.strip()  # Removes leading/trailing spaces

In [ ]:
print(data1[dependent1].isna().sum())
print(len(data1))  # Total rows


In [ ]:
print(data1[dependent1].dtype)


In [ ]:
print(data1[dependent1].head(1000))


In [ ]:
# Show the unique value of specific coloumn
data1[dependent1].unique()

**2.2.3) Data preprocessing**
such as Data formatting, Data cleaning, Data anonymization and Data sampling.

In [ ]:
# check what is the percentage of null values in each column
#print(((pd.isnull(data2).sum()/len(data2))*100).round())

In [ ]:
# Manipulate unclear values
data1 = data1.replace('?', np.nan)  # replace ? with nan
data1 = data1.replace('x', np.nan)  # replace ? with nan
data1 = data1.replace('0.0%', 0)  # replace ? with nan
data1 = data1.replace('0', 0)  # replace ? with nan
data1 = data1.replace('1', 1)  # replace ? with nan
data1 = data1.replace('2', 2)  # replace ? with nan
data1 = data1.replace('3', 3)  # replace ? with nan
data1 = data1.replace('^\s+', np.nan, regex=True)  # replace empty spaces with nan value

In [ ]:
# Handling some missing values
data1["smoker"].fillna(0, inplace=True) # notice from excel sheet that null values in smokers have value 0 in smoking
data1["alcohol_intake"].fillna(0, inplace=True) # notice from excel sheet that non alcohol drinker dont have alcoholintaker

In [ ]:
#Remove all features that contain 50% or more null
perc = 90.0
min_count = int(((100 - perc) / 100) * data1.shape[0] + 1)
data1 = data1.dropna(axis=1, thresh=min_count)  # Drop columns which contain more half than of their values are missing
print(data1.shape)  # the original shape 409 columns

In [ ]:
# Convert some categorical data to their own type
data1['t_stage'] = data1['t_stage'].astype('category')
data1['n_stage'] = data1['n_stage'].astype('category')
data1['m_stage'] = data1['m_stage'].astype('category')
data1['r_stage'] = data1['r_stage'].astype('category')
#data1['ethnicity'] = data1['ethnicity'].astype('category')
data1['Site'] = data1['Site'].astype('category')

In [ ]:
#Change all date features to the real-time date
cols= ['Smoker', 'Menopausal', 'Type_Surgery', 'Side_of_Primary', 'Histological_Type', 'Clinical_T-Stage', 'Clinical_N-Stage', 'ER_Status', 'HER-2_Status', 'PR_Status', 'Treated_breast_radio']
#for col in cols:
 # data1[col] = data1[col].astype('category')


In [ ]:
print(list(data2.columns))

In [ ]:
data1.shape

In [ ]:
data1.shape

In [ ]:
data2= pairwisecorr(data2, dependent1, threshold=0.7)

In [ ]:
data2.shape

In [ ]:
data2.shape

In [ ]:
data2 = calculate_vif(data2, 5)

In [ ]:
data2.shape

**2.2.4) Data transformation such as Scaling, Decomposition and Aggregation**

In [ ]:
# Rename the subject id column to the id in order to be mereged
data2.rename(columns = {'Subject_Id':'Subject Id'}, inplace = True)


In [ ]:
# Mapping data1 and data2 together by the id column

n_df = pd.merge(data1, data2,on = 'Subject Id', how='left')
#n_df = data1

print(n_df.shape)

In [ ]:
# I f I want to use clinic data only
#n_df= data1

In [ ]:
print(n_df.shape)

In [ ]:
print(list(n_df.columns))

In [ ]:
# Save the DataFrame to a new CSV file
#n_df.to_csv('combined.csv', index=False)

**Correct datatypes in dataframe**

In [ ]:
# Check all object-type columns
object_columns = n_df.select_dtypes(include=['object']).columns
print(f"Object-type columns: {list(object_columns)}")


In [ ]:
# based on above delete un nessery ones
n_df.drop(columns=object_columns, inplace=True)

In [ ]:
n_df['bmi'] = (n_df['weight_at_cancer_diagnosis_kg'] / n_df['height_cm'].div(1000).pow(2)).div(100)

**2.2.5) data visualising**

In [ ]:
# for visualising
import matplotlib.pyplot as plt
from matplotlib import pyplot
from matplotlib.dviread import DviFont
import seaborn as sns
# this make the shape blue
sns.set()

In [ ]:
def visualising(df, col):
  plt.subplot(121), sns.distplot(df[col])
  plt.subplot(122), df[col].plot.box(figsize=(16,5))
  plt.show()

In [ ]:
#visualising(data1, 'age')

visualising(data1, 'age')

In [ ]:
def value_percentages(df, col):
    counts = df[col].value_counts()
    percentages = df[col].value_counts(normalize=True) * 100

    # Plot
    plt.figure(figsize=(8, 6))
    ax = sns.barplot(x=percentages.index, y=percentages.values, width=0.5, palette= 'deep')

    for i, (p, c) in enumerate(zip(percentages.values, counts.values)):
        ax.annotate(f'{p:.1f}%\n(n={c})', (i, p + 1), ha='center', va='bottom', fontsize=10)

    plt.title(f'Percentage Distribution of {col}')
    plt.ylabel('Percentage')
    plt.xlabel(col)
    plt.ylim(0, max(percentages.values) + 10)
    plt.tight_layout()
    plt.show()

    #return pd.DataFrame({'Count': counts, 'Percentage': percentages})


In [ ]:
value_percentages(n_df, dependent1)

In [ ]:
# this shape detect the outlies values easily
def dots_shape(df, col1, col2, dependent):
    plt.figure(figsize=(10, 6))
    ax = sns.scatterplot(
        data=df,
        x=col1,
        y=col2,
        hue=dependent,
        palette='Set2',
        s=30,
        edgecolor='black',
        #style=dependent
    )

    plt.title(f'{col1} vs {col2} by {dependent}', fontsize=14)
    plt.xlabel(col1, fontsize=12)
    plt.ylabel(col2, fontsize=12)
    plt.legend(title=dependent, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


In [ ]:
dots_shape(n_df, 'age', 'bmi', dependent1)

In [ ]:

# the code below is used to group data in order to plot the data clearly
bins = [18, 30, 40, 50, 60, 70, 130]
labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70+']
data1['age_group'] = pd.cut(data1.age, bins, labels = labels,include_lowest = True)
data1['age_group']

bins = [10, 18.5, 25, 30, 40, 100]
labels = ['underweight', 'healthy', 'overweight', 'obesity', 'severe obesity']
data1['bmi_group'] = pd.cut(n_df.bmi, bins, labels = labels,include_lowest = True)



In [ ]:
# neww
# count distribution of dependent1 over spesific col in dataframe
def Count_distribution(df, col):
    plt.figure(figsize=(12, 7))
    ax = sns.countplot(
        x=col,
        hue=dependent1,  # make sure this is a string or passed in
        data=df,
        palette=['#00CED1', "#DC143C", '#ab6b37', 'blue'],
        width=0.8
    )

    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_ylabel("Number of Patients")
    ax.set_title(f'The Distribution of {dependent1} Patients Based on {col}')
    plt.grid(True)

    # Dynamically set y-ticks
    y_max = max([patch.get_height() for patch in ax.patches])
    step = 50  # You can change this to 100 if preferred
    plt.yticks(np.arange(0, y_max + step, step))

    plt.tight_layout()
    plt.show()


In [ ]:
Count_distribution(n_df, "surgery_axilla_type")

In [ ]:
# newwwww

def Percent_distribution(df, col, dependent):
    # Get counts and proportions
    total_counts = df[col].value_counts().sort_index()
    dependent_counts = df.groupby([col, dependent]).size().unstack(fill_value=0)
    proportions = dependent_counts.div(total_counts, axis=0).fillna(0)* 100

    # Prepare data for plotting
    plot_data = proportions.reset_index().melt(id_vars=col, var_name=dependent, value_name='proportion')

    # Create plot
    plt.figure(figsize=(14, 7))
    ax = sns.barplot(
        x=col,
        y='proportion',
        hue=dependent,
        data=plot_data,
        palette= 'deep',
        #palette=['#00CED1', "#DC143C", '#ab6b37', 'blue'],
        ci=None
    )

    # Annotate correct counts using dependent_counts
    for bar, (_, row) in zip(ax.patches, plot_data.iterrows()):
        group = row[col]
        hue_val = row[dependent]
        proportion = row['proportion']
        try:
            count = dependent_counts.loc[group, hue_val]
            if proportion > 0:
                ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.5,  # small offset above bar
                        f'n={count}\n({proportion:.1f}%)',
                        ha='center', va='bottom', fontsize=9
                        )
        except KeyError:
            continue

    # Final plot formatting
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_ylabel("Percentage")
    plt.ylim(0, plot_data['proportion'].max() + 10)
    ax.set_xlabel(f"{col}")
    ax.set_title(f'Proportion of {dependent} Values Across {col}')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
#Percent_distribution(data1, 'bmi_group', dependent1)

In [ ]:
Percent_distribution(n_df, 'education_profession', dependent1)

In [ ]:
# to confirm the result above
#n_df.groupby(['education_profession', dependent1]).size()

In [ ]:
Cuniq(n_df, dependent1)

In [ ]:
print(list(n_df.columns))

In [ ]:
#educed_df= n_df

In [ ]:
# select coloums that are date type
n_df.select_dtypes(include=['datetime'])

In [ ]:
# Drop date-type features
n_df = n_df.drop(columns=n_df.select_dtypes(include=['datetime']).columns)

**3. Model Building**

**3.1) Importing Required Libraries**

In [ ]:
# For statistic Method
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

from scipy.stats import chi2_contingency

# for calculate geometric mean
from numpy import argmax
from numpy import interp

from tabulate import tabulate

#from sklearn import feature_selection
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold

from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report

# Importing ML models
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier as KNN
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import VotingClassifier
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# Import Keras for implementing autoencoders
import keras
from imblearn.over_sampling import SMOTE



from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier # Import HistGradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold # import RepeatedStratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
import matplotlib.pyplot as plt
from tabulate import tabulate

**3.2) Choosing the Y depentent feature in order to build our model**

In [ ]:
print(list(n_df.columns))

In [ ]:
n_df.head()

In [ ]:
n_df.columns = n_df.columns.to_flat_index()

In [ ]:
Cuniq(n_df, dependent1)

From above we can see that Grade 3 atrophy is contain 24 cases, g1 and g2 in clinics have the same procedure, so thought doing binary classification is the right option for Erythema.

In [ ]:
# We transform the arm_lymphodema_24 column values to 0 and 1 for a better exploration,

reduced_df = n_df.replace(dict(breast_arm_lymphodema_24m={0: 0, 1: 1, 2:1 , 3:1}))
# When combining 2 and 3
#n_df = n_df.replace(dict(breast_oedema_24m={0: 0, 1: 1, 2:2 , 3:2}))
# if I want to predict patients who will develop only grade 1 atrophy
#reduced_df = n_df[n_df[dependent1].isin([0.0, 1.0])]

In [ ]:
# if I want to predict patients who will develop only grade 2 atrophy
#reduced_df = n_df[n_df[dependent1].isin([0.0, 2.0])]

In [ ]:
# this is needed when predicting G2
#reduced_df= reduced_df.replace(dict(breast_atrophy_24m={0: 0, 2: 1}))

In [ ]:
Cuniq(reduced_df, dependent1)

In [ ]:
# Working with dependent after transfered to dummy
'''
# the code below show that we made two level for our y values to be grade 1 and grade 0
a = pd.get_dummies(reduced_df[dependent1], prefix = dependent1)
frames = [reduced_df, a]

reduced_df.drop(dependent1, axis='columns', inplace=True) # drop the column

reduced_df = pd.concat(frames, axis = 1)
reduced_df.columns
'''

In [ ]:
# select categorical features in order to transform them to numeric data
cat_columns = reduced_df.select_dtypes(['category']).columns
# trans the values to numeric and apply that to the data
reduced_df[cat_columns] = reduced_df[cat_columns].apply(lambda x: x.cat.codes)

In [ ]:
Cuniq(reduced_df, 't_stage')

In [ ]:
print(list(reduced_df.columns))

In [ ]:
# we remove other side effect
reduced_df_clinic=reduced_df[['height_cm', 'weight_at_cancer_diagnosis_kg', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'alcohol_current_consumption', 'menopausal_status', 'monopause_age_yrs', 'hormone_replacement_therapy', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'antidiabetic', 'ace_inhibitor', 'other_antihypertensive_drug', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'antidepressant', 'breast_cancer_family_history_1st_degree', 'radiotherapy_toxicity_family_history', 'ethnicity', 'education_profession', 'household_members', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_side_of_primary', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'n_stage', 'm_stage', 'r_stage', 'ki_67_status_pc', 'er_status', 'her2_status', 'pr_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'sys_tamoxifen', 'sys_aromatase', 'sys_antiher2', 'radio_interrupted', 'radio_breast_dose_Gy', 'radio_photon_dose_MV', 'radio_breast_fractions', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_breast_internal_mammary_volume_cm3', 'radio_skin_max_dose_Gy', 'radio_skin_delineation', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_boost_type', 'radio_bolus', 'radio_boost_sequence', 'surgery_axilla_type', 'radio_breast_fractions_dose_per_fraction_Gy', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'radio_photon_boostdose_precise_Gy', 'radio_photon_boost_fractions', 'radio_photon_boost_fractions_per_week', 'radio_photon_boost_dose_per_fraction_Gy', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline', 'bmi', dependent1]]

In [ ]:
# we remove other side effect
reduced_df=reduced_df[['height_cm', 'weight_at_cancer_diagnosis_kg', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'alcohol_current_consumption', 'menopausal_status', 'monopause_age_yrs', 'hormone_replacement_therapy', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'antidiabetic', 'ace_inhibitor', 'other_antihypertensive_drug', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'antidepressant', 'breast_cancer_family_history_1st_degree', 'radiotherapy_toxicity_family_history', 'ethnicity', 'education_profession', 'household_members', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_side_of_primary', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'n_stage', 'm_stage', 'r_stage', 'ki_67_status_pc', 'er_status', 'her2_status', 'pr_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'sys_tamoxifen', 'sys_aromatase', 'sys_antiher2', 'radio_interrupted', 'radio_breast_dose_Gy', 'radio_photon_dose_MV', 'radio_breast_fractions', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_breast_internal_mammary_volume_cm3', 'radio_skin_max_dose_Gy', 'radio_skin_delineation', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_boost_type', 'radio_bolus', 'radio_boost_sequence', 'surgery_axilla_type', 'radio_breast_fractions_dose_per_fraction_Gy', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'radio_photon_boostdose_precise_Gy', 'radio_photon_boost_fractions', 'radio_photon_boost_fractions_per_week', 'radio_photon_boost_dose_per_fraction_Gy', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline', 'rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170', 'bmi',  dependent1]
]


In [ ]:
reduced_df.shape

**3.3) Feature selection:is very important to remove irrelevant features result in a better performance model, to easier understand the model and to run the model fast**




**3.3.1) Pairwise correlated feature selection**

In [ ]:
pairwisecorr(reduced_df, dependent1, 0.7) # May some feature are missing so compare with below

In [ ]:
reduced_df.shape

In [ ]:
reduced_df_clinic_c= reduced_df_clinic[['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'alcohol_current_consumption', 'menopausal_status', 'monopause_age_yrs', 'hormone_replacement_therapy', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'radiotherapy_toxicity_family_history', 'ethnicity', 'education_profession', 'household_members', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'ki_67_status_pc', 'er_status', 'her2_status', 'pr_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'sys_aromatase', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_breast_internal_mammary_volume_cm3', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_boost_type', 'radio_bolus', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'radio_photon_boostdose_precise_Gy', 'radio_photon_boost_fractions', 'radio_photon_boost_dose_per_fraction_Gy', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline',  dependent1]
]


In [ ]:
reduced_df_clinic_c.shape

In [ ]:
reduced_df_c= reduced_df[['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'alcohol_current_consumption', 'menopausal_status', 'monopause_age_yrs', 'hormone_replacement_therapy', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'radiotherapy_toxicity_family_history', 'ethnicity', 'education_profession', 'household_members', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'ki_67_status_pc', 'er_status', 'her2_status', 'pr_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'sys_aromatase', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_breast_internal_mammary_volume_cm3', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_boost_type', 'radio_bolus', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'radio_photon_boostdose_precise_Gy', 'radio_photon_boost_fractions', 'radio_photon_boost_dose_per_fraction_Gy', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline', 'rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170', dependent1]]


In [ ]:
nullP(reduced_df_clinic_c)

In [ ]:
# Select only the numeric columns
numeric_df = reduced_df.select_dtypes(include=['number'])

# Find columns with any negative values
columns_with_negatives = numeric_df.columns[(numeric_df < 0).any()]

# Create a new DataFrame with only these columns
negative_values_df = numeric_df[columns_with_negatives]

# Display the new DataFrame
print(list(negative_values_df))

In [ ]:
# Replace negative values with NaN or drop them
positive_data_df = numeric_df.where(numeric_df >= 0)

# If you also want to include non-numeric columns in the new DataFrame
positive_data_df = pd.concat([positive_data_df, reduced_df.select_dtypes(exclude=['number'])], axis=1)

# Display the new DataFrame with only positive data
print(list(positive_data_df))

In [ ]:
X_per_removed(reduced_df_clinic_c, 19)

In [ ]:
feat_cl20= ['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'menopausal_status', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'ethnicity', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'er_status', 'her2_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline']

df_cl20= reduced_df_clinic_c[feat_cl20 + [dependent1]]





In [ ]:
df_20= reduced_df_c[['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'menopausal_status', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'ethnicity', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'er_status', 'her2_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline', 'rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170', dependent1]
]



In [ ]:
feat_snp_20= ['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170']
df_snp_20= reduced_df[feat_snp_20 + [dependent1]]

In [ ]:
# litrerature snips
'''lit_snp= ['rs1037091', 'rs2211845', 'rs2836019', 'rs1662988', 'rs315721', 'rs849530', 'rs158689', 'rs3176861', 'rs1554606', 'rs1143634', 'rs1143643', 'rs17561', 'rs1800925', 'rs3775203', 'rs1056890', 'rs10464063', 'rs6879285', 'rs1130379', 'rs2239702', 'rs11801866', 'rs7330636', 'rs1985242', 'rs2070762', 'rs165656', 'rs770298', 'rs9534511']
df_lit_snp = reduced_df[lit_snp + [dependent1]]
'''

In [ ]:
# Just check in case of some missing data appeared
nullP(df_cl20)

In [ ]:
Cuniq(df_cl20, dependent1)

In [ ]:
# Applied this variable for feature selection
#k =[5, 6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,22,25,30,40]
k =[10, 15, 20, 25,30]

In [ ]:
def cross_validated_feature_selection(df, col, splits):

  # Prepare your data
  print('Original Data shape is', df.shape)
  df = df.dropna()
  X = df.drop(columns=[col])
  y = df[col]
  print('Data shape is', df.shape)
  # Optional: scale if needed
  X_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

  # Model pipeline
  pipeline = Pipeline([
      ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,bootstrap=True, random_state=5))
  ])

  # Cross-validate with AUC
  skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=0)
  auc_scores = cross_val_score(pipeline, X_scaled, y, cv=skf, scoring='roc_auc')

  print(f"✅ Mean AUC with final selected features: {auc_scores.mean():.4f}")


#First Method:Features selction

**3.3.2) Univariate Selection** — Statistical tests may be used to pick certain features that have the best relationship to the performance variable. The scikit-learn library provides the SelectKBest class that can be used to select a specific number of features in a suite of different statistical tests. The below following example uses the chi-squared (chi2) statistical test for non-negative features to select 10 of the best features from the Mobile Price Range Prediction Dataset.

## not scaled

In [ ]:
# this is my first features selection method, it is weak method because:
# Done on whole dataset, No scaling, Data Leakage, Worse performance due to Chi² needing non-negative scaled data.

def featureselection1(df, dependent, k):
    # Drop rows with missing values
    df = df.dropna(axis=0, how='any', inplace=False)

    # Separate features and target variable
    X = df.loc[:, df.columns != dependent]
    y = df[dependent]

    # Select only numeric columns (excluding object and date features)
    X = X.select_dtypes(include=['number'])

    # Ensure all feature values are non-negative (Chi-Squared requirement)
    X = X.applymap(lambda x: max(x, 0))

    # Apply SelectKBest class to extract top k best features using chi2
    bestfeatures = SelectKBest(score_func=chi2, k=k)
    fit = bestfeatures.fit(X, y)

    # Create DataFrames for scores and feature names
    dfscores = pd.DataFrame(fit.scores_, columns=['Score'])
    dfcolumns = pd.DataFrame(X.columns, columns=['TopFeat'])

    # Concatenate the DataFrames for better visualization
    featureScores = pd.concat([dfcolumns, dfscores], axis=1)

    # Sort and select the top k features
    top_features = featureScores.nlargest(k, 'Score')

    # Extract the top k feature names into a list
    top_features_list = top_features['TopFeat'].tolist()

    # Plot the top k features
    plt.figure(figsize=(10, 8))
    plt.barh(top_features['TopFeat'], top_features['Score'], color='skyblue')
    plt.xlabel('Score')
    plt.title(f'Top {k} Features selected by Chi-Squared')
    plt.gca().invert_yaxis()  # Invert y-axis for better visualization
    plt.show()  # Show the plot

    # Print the top k features
    print(tabulate(top_features, headers='keys', tablefmt='psql'))
    # Print the list of top 30 features
    print(f"\nList of top {k} features:\n", top_features_list, '\n\n')


    # Prepare data with selected features only
    X_selected = X[top_features_list]

    # Build pipeline and evaluate using cross-validation
    pipeline = Pipeline([
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,bootstrap=True, random_state=5))
    ])

    # Use StratifiedKFold and calculate mean AUC
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(pipeline, X_selected, y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC score using top {k} features: {auc_scores.mean():.4f}")
    return top_features_list, auc_scores.mean()


In [ ]:
#Cuniq(df_20, 'education_profession')


In [ ]:
#for i in k:
#  top_features_list= featureselection1(df_20, dependent1, k=i)
top_features_list= featureselection1(df_cl20, dependent1, k=20)  # df has 20% and less missing
#top_features_list= featureselection1(reduced_df_c, dependent1, k=16) # df has 50% and less missing

Takeaways: Based on the whole features (only Feat has less than 50% missing values )
* Mean AUC score using top 10 features: 0.6630
* Mean AUC score using top 11 features: 0.6900
* Mean AUC score using top 12 features: 0.6814
* Mean AUC score using top 13 features: 0.6767
* Mean AUC score using top 14 features: 0.6727
* Mean AUC score using top 15 features: 0.6938
* Mean AUC score using top 16 features: 0.7148
* Mean AUC score using top 17 features: 0.6619
* Mean AUC score using top 18 features: 0.6741
* Mean AUC score using top 19 features: 0.6660
* Mean AUC score using top 20 features: 0.6760
* Mean AUC score using top 25 features: 0.6663
* Mean AUC score using top 30 features: 0.6773


Takeaways2: Based on the features that only has less than 20% missing values:
* Mean AUC score using top 5 features: 0.6982
* Mean AUC score using top 6 features: 0.7008
* Mean AUC score using top 7 features: 0.6830
* Mean AUC score using top 8 features: 0.6805
* Mean AUC score using top 9 features: 0.6937
* Mean AUC score using top 10 features: 0.7045
* Mean AUC score using top 11 features: 0.6948
* Mean AUC score using top 12 features: 0.6991
* Mean AUC score using top 13 features: 0.7029
* Mean AUC score using top 14 features: 0.6972
* Mean AUC score using top 15 features: 0.7062
* Mean AUC score using top 16 features: 0.6933
* Mean AUC score using top 17 features: 0.7015
* Mean AUC score using top 18 features: 0.7011
* Mean AUC score using top 19 features: 0.7025
* Mean AUC score using top 20 features: 0.6967
* Mean AUC score using top 22 features: 0.7004
* Mean AUC score using top 25 features: 0.6991
* Mean AUC score using top 30 features: 0.6902

## scaled

In [ ]:
# this function same as above week performance but with scaling
# Done on whole dataset, with scaling, Data Leakage, Overestimates true performance due to leakage.

def featureselection1_scaled(df, dependent, k):
    # Drop rows with missing values
    df = df.dropna(axis=0, how='any')

    # Separate features and target
    X = df.loc[:, df.columns != dependent]
    y = df[dependent]

    # Keep only numeric features and scale for Chi2
    X = X.select_dtypes(include=['number'])
    X_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

    # Apply Chi2 feature selection
    selector = SelectKBest(score_func=chi2, k=k)
    selector.fit(X_scaled, y)

    # Score and extract top features
    scores = selector.scores_
    top_features = pd.DataFrame({
        'TopFeat': X.columns,
        'Score': scores
    }).nlargest(k, 'Score')

    selected_features = top_features['TopFeat'].tolist()

    # Plot
    plt.figure(figsize=(10, 8))
    plt.barh(top_features['TopFeat'], top_features['Score'], color='skyblue')
    plt.xlabel('Score')
    plt.title(f'Top {k} Features selected by Chi-Squared')
    plt.gca().invert_yaxis()
    plt.show()

    # Print results
    print(tabulate(top_features, headers='keys', tablefmt='psql'))
    print(f"\nList of top {k} features:\n", selected_features, '\n\n')

    # Prepare data with selected features only
    X_selected = X_scaled[selected_features]

    # Build pipeline and evaluate using cross-validation
    pipeline = Pipeline([
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,bootstrap=True, random_state=5))
    ])

    # Use StratifiedKFold and calculate mean AUC
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(pipeline, X_selected, y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC score using top {k} features: {auc_scores.mean():.4f}")
    return selected_features, auc_scores.mean()


In [ ]:
#for i in k:
#  top_Scaled_features_list= featureselection1_scaled(df_20, dependent1, k=i)
top_scaled_feat= featureselection1_scaled(df_20, dependent1, k=20)

Takeaways3: After scaling Based on the features that only has less than 20% missing values:
* Mean AUC score using top 5 features: 0.6776
* Mean AUC score using top 6 features: 0.6601
* Mean AUC score using top 7 features: 0.6549
* Mean AUC score using top 8 features: 0.6451
* Mean AUC score using top 9 features: 0.6803
* Mean AUC score using top 10 features: 0.7059
* Mean AUC score using top 11 features: 0.7136
* Mean AUC score using top 12 features: 0.7213
* Mean AUC score using top 13 features: 0.7145
* Mean AUC score using top 14 features: 0.7159
* Mean AUC score using top 15 features: 0.7110
* Mean AUC score using top 16 features: 0.7063
* Mean AUC score using top 17 features: 0.7081
* Mean AUC score using top 18 features: 0.7110
* Mean AUC score using top 19 features: 0.7098
* Mean AUC score using top 20 features: 0.7114
* Mean AUC score using top 22 features: 0.7112
* Mean AUC score using top 25 features: 0.7078
* Mean AUC score using top 30 features: 0.7088

## inside CV

In [ ]:
# ✅ Inside CV folds	✅ Yes With scaling	✅ Clean no leakage	📉 True estimate of performance

def featureselection1_pipeline_auc(df, dependent, k):
    df = df.dropna()

    X = df.loc[:, df.columns != dependent].select_dtypes(include=['number'])
    y = df[dependent]

    # Build pipeline with scaling, feature selection, and classification
    pipeline = Pipeline([
        ('scaler', MinMaxScaler()),
        ('selectk', SelectKBest(score_func=chi2, k=k)),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,bootstrap=True, random_state=5))
    ])

    # Cross-validation setup
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC score using top {k} features (Inside CV): {auc_scores.mean():.2f}")
    return auc_scores.mean()


In [ ]:
#featureselection1_pipeline_auc(df_cl20, dependent1, k=20)
for i in k:
  top_Scaled_pipeline_features_list= featureselection1_pipeline_auc(df_20, dependent1, k=i)

* Mean AUC score using top 5 features (no leakage): 0.6727
* Mean AUC score using top 6 features (no leakage): 0.6584
* Mean AUC score using top 7 features (no leakage): 0.6597
* Mean AUC score using top 8 features (no leakage): 0.6742
* Mean AUC score using top 9 features (no leakage): 0.6701
* Mean AUC score using top 10 features (no leakage): 0.6566
* Mean AUC score using top 11 features (no leakage): 0.6592
* Mean AUC score using top 12 features (no leakage): 0.6909
* Mean AUC score using top 13 features (no leakage): 0.6940
* Mean AUC score using top 14 features (no leakage): 0.7001
* Mean AUC score using top 15 features (no leakage): 0.6942
* Mean AUC score using top 16 features (no leakage): 0.6950
* Mean AUC score using top 17 features (no leakage): 0.7114
* Mean AUC score using top 18 features (no leakage): 0.6964
* Mean AUC score using top 19 features (no leakage): 0.6981
* Mean AUC score using top 20 features (no leakage): 0.6999
* Mean AUC score using top 22 features (no leakage): 0.6941
* Mean AUC score using top 25 features (no leakage): 0.6904
* Mean AUC score using top 30 features (no leakage): 0.6931
* Mean AUC score using top 40 features (no leakage): 0.6908

**Takeaways4,** in above analysis this is a real and true feature selection as only happens in the training set,,, But there is an issue because  it is not only model building I need to look at the important features and observe their importance.

## features frequency

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

def track_feature_selection_frequency_with_auc(df, dependent, k, n_splits):
    df = df.dropna(axis=0, how='any')

    X = df.loc[:, df.columns != dependent]
    y = df[dependent]

    # Use only numeric features
    X = X.select_dtypes(include=['number'])

    # Scale features (Chi2 needs non-negative values)
    X_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    feature_counter = Counter()
    auc_scores = []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_scaled, y)):
        X_train, y_train = X_scaled.iloc[train_index], y.iloc[train_index]
        X_test, y_test = X_scaled.iloc[test_index], y.iloc[test_index]

        # Apply Chi2 on training fold
        selector = SelectKBest(score_func=chi2, k=k)
        selector.fit(X_train, y_train)

        selected_features = X_train.columns[selector.get_support()]

        # Count selected features
        feature_counter.update(selected_features)

        # Predict probabilities on the test fold
        X_test_selected = X_test[selected_features]
        # Here we use a simple logistic regression as the classifier, you can use another classifier if needed
        clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,bootstrap=True, random_state=5)
        clf.fit(X_train[selected_features], y_train)

        # Get predicted probabilities for AUC calculation
        y_prob = clf.predict_proba(X_test_selected)[:, 1]

        # Calculate AUC score for the fold
        auc = roc_auc_score(y_test, y_prob)
        auc_scores.append(auc)

        #print(f"Fold {fold_idx+1}: Selected features: {list(selected_features)} | AUC: {auc:.4f}")
        print(f"{list(selected_features)}")

    # Calculate the mean AUC
    mean_auc = np.mean(auc_scores)
    print(f"Mean AUC across {n_splits} folds: {mean_auc:.4f}")

    # Convert to DataFrame
    freq_df = pd.DataFrame.from_dict(feature_counter, orient='index', columns=['Frequency'])
    freq_df = freq_df.sort_values(by='Frequency', ascending=False)

    # Plot feature frequency
    plt.figure(figsize=(10, 12))
    sns.barplot(x='Frequency', y=freq_df.index, data=freq_df, palette='magma')
    plt.xlabel('Selection Frequency')
    plt.ylabel('KBest Features')
    plt.title(f'Chi2 Feature Selection Frequency Across {n_splits} Folds (Top {k} features/fold)')
    plt.xticks(np.arange(0, n_splits+1, 1))
    plt.show()

    print(tabulate(freq_df, headers='keys', tablefmt='psql'))
    return freq_df, mean_auc


In [ ]:
# with this method getting higher auc than the one with most frequency
freq_df_auc = track_feature_selection_frequency_with_auc(df_snp_20, dependent1, 20, n_splits=10) # we can use reduced_df_c, or df_20


In [ ]:
# the version that returns most frequent features across folds

def track_feature_selection_frequency(df, dependent, k, n_splits):
    df = df.dropna(axis=0, how='any')

    X = df.loc[:, df.columns != dependent]
    y = df[dependent]

    # Use only numeric features
    X = X.select_dtypes(include=['number'])

    # Scale features (Chi2 needs non-negative values)
    X_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    feature_counter = Counter()

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X_scaled, y)):
        X_train, y_train = X_scaled.iloc[train_index], y.iloc[train_index]

        # Apply Chi2 on training fold
        selector = SelectKBest(score_func=chi2, k=k)
        selector.fit(X_train, y_train)

        selected_features = X_train.columns[selector.get_support()]

        # Count selected features
        feature_counter.update(selected_features)

        print(f"Fold {fold_idx+1}: Selected features: {list(selected_features)}")

    # Convert to DataFrame
    freq_df = pd.DataFrame.from_dict(feature_counter, orient='index', columns=['Frequency'])
    freq_df = freq_df.sort_values(by='Frequency', ascending=False)

    # Plot
    plt.figure(figsize=(10, 12))

    sns.barplot(x='Frequency', y=freq_df.index, data=freq_df, palette='magma')
    plt.xlabel('Selection Frequency')
    plt.ylabel('KBest Features')
    plt.title(f'Chi2 Feature Selection Frequency Across {n_splits} Folds (Top {k} features/fold)')
    plt.xticks(np.arange(0, n_splits+1, 1))
    #plt.gca().invert_yaxis()
    plt.show()
    print(tabulate(freq_df, headers='keys', tablefmt='psql'))
    return freq_df


In [ ]:
#track_feature_selection_frequency(reduced_df_c, dependent1, 10, n_splits=10)
# Step 1: Track feature selection frequency
freq_df = track_feature_selection_frequency(df_20, dependent1, 40, n_splits=10) # we can use reduced_df_c, or df_20

# Step 2: Choose top N most frequently selected features
N = 20  # or any number based on your needs , WHEN SELECTED 26 GOT ONLY ONE SNP
top_features_final = freq_df.head(N).index.tolist()

print("🎯 Final Selected Features Based on Frequency:")
print(top_features_final)

# Step 3: Create final dataframe with top features + y column
df_selectedBest = df_20[top_features_final + [dependent1]]


In [ ]:
X_per_removed(df_selectedBest, 9)

In [ ]:
# use this to remove features with highly missing and keep as possible as i can
df_selectedBest2= ['age', 'bra_cup_size', 'smoker', 'other_collagen_vascular_disease', 'diabetes', 'ace_inhibitor', 'hypertension', 'tumour_side_of_primary', 'delayed_healing', 'on_statin', 'other_lipid_lowering_drugs', 'analgesics', 'ethnicity', 'surgery_type', 'post_operative_haematoma', 'radio_heart_delineation', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_nipple_retraction_baseline', 'Site',   dependent1]


In [ ]:
cross_validated_feature_selection(df_selectedBest, dependent1, splits=10)

# Second feature selectin Method

**3.3.3) Feature Importance** — You can gain the significance of each feature of your dataset by
using the Model Characteristics property. Feature value gives you a score for every function of your results, the higher the score the more significant or appropriate the performance variable is. Feature importance is the built-in class that comes with Tree Based Classifiers, we will use the Extra Tree Classifier to extract the top 10 features for the dataset.

## outside CV

In [ ]:
def featureselection2(df, col, k):
    # Plot the top k features
    plt.figure(figsize=(10, 10))

    # Initialize the model
    model = ExtraTreesClassifier(random_state=10)

    # Drop rows with missing values
    df = df.dropna(axis=0, how='any', inplace=False)

    # Split data into features (X) and target (y)
    X = df.loc[:, df.columns != col]
    y = df[col]

    # Select only numeric columns (excluding object and date features)
    X = X.select_dtypes(include=['number'])
    # Fit the model
    model.fit(X, y)

    # Get feature importances
    feat_importances = pd.Series(model.feature_importances_, index=X.columns)

    # Select the top k important features
    top_features = feat_importances.nlargest(k)

    # Plot the top k features
    top_features.plot(kind='barh')
    #sns.barplot(x=top_features, y=feat_importances, data=df, palette='viridis')
    plt.xlabel("Feature Importance")
    plt.title(f"Top {k} Important Features")
    plt.show()
    print(f"\n\n Important {k} features:", top_features.index.tolist())

    # Prepare data with selected features only
    X_selected = X[top_features.index.tolist()]
    # Build pipeline and evaluate using cross-validation
    pipeline = Pipeline([
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,bootstrap=True, random_state=5))
    ])

    # Use StratifiedKFold and calculate mean AUC
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(pipeline, X_selected, y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC score using top {k} features: {auc_scores.mean():.4f}")
    return top_features.index.tolist(), auc_scores.mean()



In [ ]:
# Run the function and store the result in top_features
featureselection2(df_cl20, dependent1, k=20)
#for i in k:
#  top_features_list2= featureselection2(reduced_df_c, dependent1, k=i)

Takeaways4: These based on reduced_df_c dataframe
* Mean AUC score using top 5 features: 0.7104
* Mean AUC score using top 6 features: 0.7586
* Mean AUC score using top 7 features: 0.7564
* Mean AUC score using top 8 features: 0.6880
* Mean AUC score using top 9 features: 0.6999
* Mean AUC score using top 10 features: 0.7240
* Mean AUC score using top 11 features: 0.7326
* Mean AUC score using top 12 features: 0.7162
* Mean AUC score using top 13 features: 0.7078
* Mean AUC score using top 14 features: 0.6982
* Mean AUC score using top 15 features: 0.7074
* Mean AUC score using top 16 features: 0.6920
* Mean AUC score using top 17 features: 0.7000
* Mean AUC score using top 18 features: 0.7098
* Mean AUC score using top 19 features: 0.7248
* Mean AUC score using top 20 features: 0.7052
* Mean AUC score using top 22 features: 0.7337
* Mean AUC score using top 25 features: 0.6813
* Mean AUC score using top 30 features: 0.6699


In [ ]:
featureselection2(df_20, dependent1, k=20)

#for i in k:
#  top_features_list2= featureselection2(df_20, dependent1, k=i)

Takeaways4: These based on df_20 dataframe
* Mean AUC score using top 5 features: 0.7205
* Mean AUC score using top 6 features: 0.7278
* Mean AUC score using top 7 features: 0.7259
* Mean AUC score using top 8 features: 0.7247
* Mean AUC score using top 9 features: 0.7225
* Mean AUC score using top 10 features: 0.7113
* Mean AUC score using top 11 features: 0.7136
* Mean AUC score using top 12 features: 0.7197
* Mean AUC score using top 13 features: 0.7114
* Mean AUC score using top 14 features: 0.7159
* Mean AUC score using top 15 features: 0.7074
* Mean AUC score using top 16 features: 0.7104
* Mean AUC score using top 17 features: 0.7223
* Mean AUC score using top 18 features: 0.7221
* Mean AUC score using top 19 features: 0.7163
* Mean AUC score using top 20 features: 0.7130
* Mean AUC score using top 22 features: 0.7116
* Mean AUC score using top 25 features: 0.7047
* Mean AUC score using top 30 features: 0.7038
* Mean AUC score using top 40 features: 0.6995

## inside CV

In [ ]:
# ✅ Inside CV folds	✅ Yes With scaling	✅ Clean no leakage	📉 True estimate of performance
from sklearn.feature_selection import SelectFromModel

def featureselection2_pipeline_auc(df, dependent, k):
    df = df.dropna()

    X = df.loc[:, df.columns != dependent].select_dtypes(include=['number'])
    y = df[dependent]

    # Pipeline: scale + feature selection + classifier
    selector_model = ExtraTreesClassifier(random_state=10)
    feature_selector = SelectFromModel(selector_model, threshold=-np.inf, max_features=k, prefit=False)

    # Build pipeline with scaling, feature selection, and classification
    pipeline = Pipeline([
        ('features importance', feature_selector),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=5))
    ])

    # Cross-validation setup
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC score using top {k} features (Inside CV): {auc_scores.mean():.2f}")
    return auc_scores.mean()


In [ ]:
#featureselection2_pipeline_auc(df_20, dependent1, k=20)
for i in k:
  top_Scaled_pipeline_features_list= featureselection2_pipeline_auc(df_20, dependent1, k=i)

## most frequency

In [ ]:
Cuniq(reduced_df_Mix, dependent1)

In [ ]:
# WITH AUC
def featureselection2_freq_withAUC(df, col, k, n_splits):
    #df = df.dropna()
    df = df.dropna(subset=[col])

    # Prepare X and y
    X = df.drop(columns=[col]).select_dtypes(include=['number'])
    y = df[col]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

    feature_counts = Counter()
    auc_scores = []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_index], y.iloc[train_index]
        X_test, y_test = X.iloc[test_index], y.iloc[test_index]

        # Feature selection model
        selector_model = ExtraTreesClassifier(random_state=10)
        selector_model.fit(X_train, y_train)

        # Select top k features based on importance
        importances = pd.Series(selector_model.feature_importances_, index=X_train.columns)
        top_features = importances.nlargest(k).index.tolist()

        # Count selected features
        feature_counts.update(top_features)
        print(f"Fold {fold_idx+1}: Selected features: {list(top_features)}")

        # **Ensure that the selected features are present in the test set**
        # **by using the intersection of features between training and testing sets**
        common_features = list(set(top_features) & set(X_test.columns))
        X_test_selected = X_test[common_features]
        X_train_selected = X_train[common_features]  # Also select for training

        # Re-train the model using only the common features
        selector_model.fit(X_train_selected, y_train)

        # Predict probabilities on the test set for AUC calculation
        y_pred_prob = selector_model.predict_proba(X_test_selected)[:, 1]  # Get the probability for the positive class

        # Compute AUC score for the current fold
        auc = roc_auc_score(y_test, y_pred_prob)
        auc_scores.append(auc)
        print(f"Fold {fold_idx+1}: AUC = {auc:.4f}")

    # Calculate the mean AUC across all folds
    mean_auc = np.mean(auc_scores)
    print(f"Mean AUC across {n_splits} folds: {mean_auc:.4f}")

    # Convert to DataFrame
    freq_df = pd.DataFrame.from_dict(feature_counts, orient='index', columns=['Frequency'])
    freq_df = freq_df.sort_values(by='Frequency', ascending=False)

    # Plot feature frequency
    plt.figure(figsize=(10, 12))
    sns.barplot(x='Frequency', y=freq_df.index, data=freq_df, palette='viridis')
    plt.xlabel('Selection Frequency')
    plt.ylabel('Feature Importances with ETC')
    plt.title(f'Feature Importances Frequency Across {n_splits} Folds (Top {k} features/fold)')
    plt.xticks(np.arange(0, n_splits+1, 1))
    plt.show()

    # Print the frequency table
    print(tabulate(freq_df, headers='keys', tablefmt='psql'))

    return freq_df, mean_auc

In [ ]:
freq2_df_auc = featureselection2_freq_withAUC(df_20, dependent1, 20, n_splits=10) # we can us

In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from tabulate import tabulate

def featureselection2_freq_withAUROC(df, col, k, n_splits):
    #df = df.dropna()
    df = df.dropna(subset=[col])

    X = df.drop(columns=[col]).select_dtypes(include=['number'])
    y = df[col]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

    feature_counts = Counter()
    auc_scores = []
    roc_data = []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_index], y.iloc[train_index]
        X_test, y_test = X.iloc[test_index], y.iloc[test_index]

        selector_model = ExtraTreesClassifier(random_state=10)
        selector_model.fit(X_train, y_train)

        importances = pd.Series(selector_model.feature_importances_, index=X_train.columns)
        top_features = importances.nlargest(k).index.tolist()

        feature_counts.update(top_features)
        print(f"Fold {fold_idx+1}: Selected features: {list(top_features)}")

        common_features = list(set(top_features) & set(X_test.columns))
        X_test_selected = X_test[common_features]
        X_train_selected = X_train[common_features]

        selector_model.fit(X_train_selected, y_train)

        y_pred_prob = selector_model.predict_proba(X_test_selected)[:, 1]
        auc = roc_auc_score(y_test, y_pred_prob)
        auc_scores.append(auc)
        print(f"Fold {fold_idx+1}: AUC = {auc:.4f}")

        fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
        roc_data.append((fpr, tpr, auc))

    # Mean and Std of AUCs
    mean_auc = np.mean(auc_scores)
    std_auc = np.std(auc_scores)
    print(f"Mean AUC across {n_splits} folds: {mean_auc:.4f} ± {std_auc:.4f}")

    # Compute mean ROC curve
    mean_fpr = np.linspace(0, 1, 100)
    tprs = []

    for fpr, tpr, _ in roc_data:
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)

    mean_tpr = np.mean(tprs, axis=0)
    std_tpr = np.std(tprs, axis=0)
    mean_tpr[-1] = 1.0

    # Plot all ROC curves + mean curve
    plt.figure(figsize=(8, 6))
    for i, (fpr, tpr, auc) in enumerate(roc_data):
        plt.plot(fpr, tpr, lw=1, alpha=0.5, label=f"Fold {i+1} (AUC = {auc:.2f})")

    plt.plot(mean_fpr, mean_tpr, color='b', label=f"Mean ROC (AUC = {mean_auc:.2f} ± {std_auc:.2f})", lw=2)
    plt.fill_between(mean_fpr, mean_tpr - std_tpr, mean_tpr + std_tpr, color='blue', alpha=0.2, label='±1 std. dev.')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves for Each Fold and Mean ROC")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Feature frequency plot
    freq_df = pd.DataFrame.from_dict(feature_counts, orient='index', columns=['Frequency'])
    freq_df = freq_df.sort_values(by='Frequency', ascending=False)

    plt.figure(figsize=(10, 12))
    sns.barplot(x='Frequency', y=freq_df.index, data=freq_df, palette='viridis')
    plt.xlabel('Selection Frequency')
    plt.ylabel('Feature Importances with ETC')
    plt.title(f'Feature Importances Frequency Across {n_splits} Folds (Top {k} features/fold)')
    plt.xticks(np.arange(0, n_splits+1, 1))
    plt.tight_layout()
    plt.show()

    print(tabulate(freq_df, headers='keys', tablefmt='psql'))

    return freq_df, mean_auc


In [ ]:
featureselection2_freq_withAUROC(reduced_df_Mixsnp, dependent1, 20, n_splits=10)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from collections import Counter
import numpy as np
import pandas as pd

def featureselection2_freq(df, col, k, n_splits):
    df = df.dropna()

    # Prepare X and y
    X = df.drop(columns=[col]).select_dtypes(include=['number'])
    y = df[col]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

    feature_counts = Counter()
    auc_scores = []

    for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train, y_train = X.iloc[train_index], y.iloc[train_index]

        # Feature selection model
        selector_model = ExtraTreesClassifier(random_state=10)
        selector_model.fit(X_train, y_train)

        # Select top k features
        importances = pd.Series(selector_model.feature_importances_, index=X_train.columns)
        top_features = importances.nlargest(k).index.tolist()

        # Count selected features
        feature_counts.update(top_features)
        print(f"Fold {fold_idx+1}: Selected features: {list(top_features)}")

   # Convert to DataFrame
    freq_df = pd.DataFrame.from_dict(feature_counts, orient='index', columns=['Frequency'])
    freq_df = freq_df.sort_values(by='Frequency', ascending=False)

    # Plot
    plt.figure(figsize=(10, 12))
    sns.barplot(x='Frequency', y=freq_df.index, data=freq_df, palette='viridis')
    plt.xlabel('Selection Frequency')
    plt.ylabel('Feature Importances with ETC')
    plt.title(f'Feature Importances Frequency Across {n_splits} Folds (Top {k} features/fold)')
    plt.xticks(np.arange(0, n_splits+1, 1))
    #plt.gca().invert_yaxis()
    plt.show()

    print(tabulate(freq_df, headers='keys', tablefmt='psql'))
    return freq_df

In [ ]:
#track_feature_selection_frequency(reduced_df_c, dependent1, 10, n_splits=10)
# Step 1: Track feature selection frequency
freq2_df = featureselection2_freq(df_20, dependent1, 40, n_splits=10) # we can use reduced_df_c, or df_20

# Step 2: Choose top N most frequently selected features
N = 20  # or any number based on your needs , WHEN SELECTED 26 GOT ONLY ONE SNP fro feature selection 1 method (chi2)
top_features_final = freq2_df.head(N).index.tolist()

print("🎯 Final Selected Features Based on Frequency:")
print(top_features_final)

# Step 3: Create final dataframe with top features + y column
df_Top_selected = df_20[top_features_final + [dependent1]]


In [ ]:
X_per_removed(df_Top_selected, 15)

In [ ]:
#df_Top_selected = df_20[top_features_final + [dependent1]]
cross_validated_feature_selection(df_Top_selected, dependent1, splits=10)

# Third feature selection method

**3.3.4 Recursive Feature Elimination (RFE)**

---


Recursive Feature Elimination (RFE) is a feature selection method used in machine learning to identify the most relevant features for a predictive model. It works by recursively removing less important features and building the model on the remaining features to determine their importance.

## outside CV

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer, roc_auc_score
# This method ranking for example 20 features as rank 1 and other varies if set the features to 20 and so on

def rfe_feature_selection3(df, target_col, n_features):

    model = RandomForestClassifier(random_state=10)

    # Drop rows with missing values
    df = df.dropna()


    df = df.dropna()

    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Encode target if it's categorical
    if y.dtype == 'object':
        y = LabelEncoder().fit_transform(y)

    # Split once for RFE
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=10)

    # RFE for top features
    rfe = RFE(estimator=model, n_features_to_select=n_features)
    rfe.fit(X_train, y_train)

    selected_features = X.columns[rfe.support_].tolist()
    print(f"\nSelected Features: {selected_features}")

    # Cross-validated AUC on selected features
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(model, X[selected_features], y, cv=skf, scoring='roc_auc')

    #print(f"Cross-Validated AUC Scores: {auc_scores}")
    print(f"Mean AUC: {auc_scores.mean():.4f} | Std: {auc_scores.std():.4f}")

    # Feature ranking
    feature_ranking = pd.DataFrame({
        'Feature': X.columns,
        'Ranking': rfe.ranking_
    }).sort_values(by='Ranking')

    # Plot the feature ranking
    plt.figure(figsize=(10, 20))
    sns.barplot(
        x='Ranking',
        y='Feature',
        data=feature_ranking,
        palette='viridis',
        order=feature_ranking.sort_values('Ranking')['Feature']
    )
    plt.title("Feature Ranking (RFE)")
    plt.xlabel("RFE Ranking (1 = Most Important)")
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()

    return selected_features, auc_scores.mean(), feature_ranking


In [ ]:
rfe_feature_selection3(df_cl20, dependent1, 20)
#for i in k:
#  rfe_feature_selection3(df_20, dependent1, i)

Takeaway6:
* Mean AUC score using top 5 features: 0.5276
* Mean AUC score using top 6 features: 0.5595
* Mean AUC score using top 7 features: 0.5658
* Mean AUC score using top 8 features: 0.6740
* Mean AUC score using top 9 features: 0.6565
* Mean AUC score using top 10 features: 0.6793
* Mean AUC score using top 11 features: 0.6813
* Mean AUC score using top 12 features: 0.6843
* Mean AUC score using top 13 features: 0.6814
* Mean AUC score using top 14 features: 0.6789
* Mean AUC score using top 15 features: 0.7059
* Mean AUC score using top 16 features: 0.6978
* Mean AUC score using top 17 features: 0.6916
* Mean AUC score using top 18 features: 0.7046
* Mean AUC score using top 19 features: 0.6980
* Mean AUC score using top 20 features: 0.6968
* Mean AUC score using top 22 features: 0.6961
* Mean AUC score using top 25 features: 0.7038
* Mean AUC score using top 30 features: 0.7048
* Mean AUC score using top 40 features: 0.7115
* Mean AUC score using top 45 features: 0.6961

## With heatmap

In [ ]:
#whith heatmap
def rfe_feature_selection_with_heatmap(df, target_col, n_features):

    model = RandomForestClassifier(random_state=10)

    # Drop rows with missing values
    df = df.dropna()

    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Encode target if it's categorical
    if y.dtype == 'object':
        y = LabelEncoder().fit_transform(y)

    # Split once for RFE
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=10)

    # RFE for top features
    rfe = RFE(estimator=model, n_features_to_select=n_features)
    rfe.fit(X_train, y_train)

    # Get selected features
    selected_features = X.columns[rfe.support_].tolist()
    print(f"\nSelected Features: {selected_features}")

    # Cross-validated AUC on selected features
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(model, X[selected_features], y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC: {auc_scores.mean():.4f} | Std: {auc_scores.std():.4f}")

    # Feature ranking for selected features
    feature_ranking = pd.DataFrame({
        'Feature': X.columns,
        'Ranking': rfe.ranking_
    }).sort_values(by='Ranking')


    # Set Feature as index (for nicer heatmap)
    feature_ranking = feature_ranking.set_index('Feature')

    # Normalize rankings to 0-1 range for better color scaling (optional)
    feature_ranking['Ranking_norm'] = (feature_ranking['Ranking'] - feature_ranking['Ranking'].min()) / (feature_ranking['Ranking'].max() - feature_ranking['Ranking'].min())

    # Plot heatmap
    plt.figure(figsize=(8, len(feature_ranking) * 0.2))  # dynamic height
    sns.heatmap(
        feature_ranking[['Ranking_norm']],
        cmap='viridis',  # reverse viridis (dark = important)
        cbar_kws={'label': 'Normalized RFE Ranking'},
        linewidths=0.5
    )

    plt.title('RFE Feature Rankings (Heatmap)')
    plt.xlabel('Normalized Ranking')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()
    return selected_features, auc_scores.mean(), feature_ranking

In [ ]:
#rfe_feature_selection_with_heatmap(reduced_df_c, dependent1, 20)

## inside CV

In [ ]:

def run_pipeline_with_feature_selection3(df, target_column, k):

    # Drop rows with missing values
    df = df.dropna()

    # Separate features and target
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Keep only numeric features (Chi2 requires non-negative numbers)
    X = X.select_dtypes(include=['number'])
    rfe_classifier = RandomForestClassifier(random_state=10)
    rfe_selector = RFE(estimator=rfe_classifier, n_features_to_select=k)
    # Define the pipeline

    pipeline_rfe = Pipeline([
    ('rfe', rfe_selector),                    # Apply RFE
    ('classifier', rfe_classifier)             # Train the model
])

    # Use Stratified K-Fold CV for balanced classes
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    scores = cross_val_score(pipeline_rfe, X, y, cv=cv, scoring='roc_auc')

    #print(f"Cross-validated accuracy scores: {scores}")
    print(f"Mean AUC score using top {k} features (Inside CV): {scores.mean():.2f}")
    return scores.mean()


In [ ]:
for i in k:
  run_pipeline_with_feature_selection3(df_20, dependent1, i)

## Most frequency

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

def featureselection3_freq(df, target_col, n_features, n_splits):
    model = RandomForestClassifier(random_state=10)

    # Drop rows with missing values
    df = df.dropna()

    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Encode target if it's categorical
    if y.dtype == 'object':
        y = LabelEncoder().fit_transform(y)

    # Prepare CV
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

    auc_scores = []
    selected_features_list = []
    feature_counts = Counter()
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

        # Use .iloc to select rows based on integer positions
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Fit RFE only on training data
        rfe = RFE(estimator=model, n_features_to_select=n_features)
        rfe.fit(X_train, y_train)

        selected_features = X_train.columns[rfe.support_].tolist()
        selected_features_list.append(selected_features)

        # Train model on selected features
        model.fit(X_train[selected_features], y_train)

        # Predict and compute AUC on test set
        y_pred_proba = model.predict_proba(X_test[selected_features])[:, 1]
        auc = roc_auc_score(y_test, y_pred_proba)
        auc_scores.append(auc)
        feature_counts.update(selected_features)
        print(f"Fold {fold+1}: AUC = {auc:.4f} | Selected Features: {selected_features}")

    print(f"\nMean AUC: {np.mean(auc_scores):.4f} | Std: {np.std(auc_scores):.4f}")
    # Convert to DataFrame
    freq_df = pd.DataFrame.from_dict(feature_counts, orient='index', columns=['Frequency'])
    freq_df = freq_df.sort_values(by='Frequency', ascending=False)

    # Plot
    plt.figure(figsize=(10, 12))
    sns.barplot(x='Frequency', y=freq_df.index, data=freq_df, palette='cividis')
    plt.xlabel('Selection Frequency')
    plt.ylabel('Feature Importances with RFE')
    plt.title(f'Feature Importances Frequency Across {n_splits} Folds (Top {n_features} features/fold)')
    plt.xticks(np.arange(0, n_splits+1, 1))
    #plt.gca().invert_yaxis()
    plt.show()

    print(tabulate(freq_df, headers='keys', tablefmt='psql'))
    return freq_df
    #return auc_scores, selected_features_list

In [ ]:
featureselection3_freq(df_20, dependent1, 20, n_splits=10)

In [ ]:
#track_feature_selection_frequency(reduced_df_c, dependent1, 10, n_splits=10)
# Step 1: Track feature selection frequency
freq3_df = featureselection3_freq(df_cl20, dependent1, 40, n_splits=10) # we can use reduced_df_c, or df_20

# Step 2: Choose top N most frequently selected features
N = 20  # or any number based on your needs , WHEN SELECTED 26 GOT ONLY ONE SNP fro feature selection 1 method (chi2)
top_features_final = freq3_df.head(N).index.tolist()

print("🎯 Final Selected Features Based on Frequency:")
print(top_features_final)

# Step 3: Create final dataframe with top features + y column
df_rfe_selected = df_cl20[top_features_final + [dependent1]]


In [ ]:
df_Top_selected= df_cl20[['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'tumour_quadrant', 'tumour_histological_grade', 'tumour_size_mm', 't_stage', 'radio_breast_fractions', 'radio_breast_ct_volume_cm3', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_ipsilateral_lung_mean_Gy', 'Site', 'breast_atrophy_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_atrophy_24m']



]


In [ ]:
X_per_removed(df_Top_selected, 9)

In [ ]:
df_rfe_selected9= df_cl20[['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'nodes_involved', 'nodes_examined', 'tumour_quadrant', 'tumour_histological_grade', 'tumour_size_mm', 't_stage', 'radio_breast_fractions', 'radio_breast_ct_volume_cm3', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_ipsilateral_lung_mean_Gy', 'Site', 'breast_atrophy_baseline', 'breast_skin_induration_tumour_bed_baseline']

]

In [ ]:
cross_validated_feature_selection(df_rfe_selected, dependent1, splits=10)

Now we tried to select some features from these feature selection, taking the features that have less null values.

In [ ]:
# befor we applied the AE we need to remove feature that have for example 20% missing

X_per_removed(reduced_df_c, 20)

# Feature Selection 4

**3.3.4) autoencoder selection method**

In [ ]:
# these features prodiced after removing above 20 % missing values
AE_features= ['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'menopausal_status', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'ethnicity', 'education_profession', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'er_status', 'her2_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline', 'rs145201352', 'rs146489859', 'rs2091891', 'rs2170306', 'rs62036939', 'rs7198421', 'rs73192930', 'rs73192932', 'rs73192934', 'rs73478102', 'rs73478106', 'rs73478113', 'rs73478121', 'rs73478127', 'rs73478152', 'rs73478154', 'rs73478167', 'rs73478176', 'rs73478181', 'rs74711328', 'rs7645688', 'rs79079523', 'rs9537701', 'rs9893368', 'breast_atrophy_24m']

 # when I selected best 20 using AE got 66 or 67 as the highest , which means that genetics improve the prediction

AE_features_clinic= ['height_cm', 'age', 'bra_cup_size', 'band_size', 'smoker', 'alcohol_intake', 'menopausal_status', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'ethnicity', 'education_profession', 'Sequence Num_x', 'surgery', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_histological_grade', 'tumour_histological_type', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'er_status', 'her2_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'Site', 'Sequence Num_y', 'Sequence Num_baseline', 'breast_atrophy_baseline', 'breast_nipple_retraction_baseline', 'breast_oedema_baseline', 'breast_skin_ulceration_baseline', 'breast_telangiectasia_tumour_bed_baseline', 'breast_telangiectasia_outside_tumour_bed_baseline', 'breast_skin_induration_tumour_bed_baseline', 'breast_skin_induration_outside_tumour_bed_baseline', 'breast_erythema_baseline', 'breast_arm_lymphodema_baseline', 'breast_skin_hyperpigmentation_baseline', 'breast_pneumontis_baseline', 'breast_pain_baseline', 'breast_swollen_arm_baseline', 'breast_atrophy_24m']  # when I selected best 20 using AE got 62--67 as the highest

# these features
AE_features2= ['height_cm', 'age', 'bra_cup_size', 'smoker', 'alcohol_intake', 'menopausal_status', 'diabetes', 'history_of_heart_disease', 'ra', 'systemic_lupus_erythematosus', 'other_collagen_vascular_disease', 'hypertension', 'depression', 'ace_inhibitor', 'on_statin', 'other_lipid_lowering_drugs', 'amiodarone', 'analgesics', 'breast_cancer_family_history_1st_degree', 'surgery_type', 'nodes_involved', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'tumour_size_mm', 't_stage', 'm_stage', 'r_stage', 'er_status', 'her2_status', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'sys_treatment', 'radio_interrupted', 'radio_photon_dose_MV', 'radio_breast_ct_volume_cm3', 'radio_breast_delineation', 'radio_skin_max_dose_Gy', 'radio_heart_mean_dose_Gy', 'radio_heart_delineation', 'radio_hot_spots_107', 'radio_ipsilateral_lung_mean_Gy', 'radio_imrt', 'radio_treatment_pos', 'radio_3d', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_boost', 'radio_breast_fractions_per_week', 'radio_photon_2nd', 'Site', 'breast_atrophy_baseline',  'breast_pain_baseline', 'breast_swollen_arm_baseline', 'rs145201352', 'rs146489859', 'rs2091891', 'rs2170306', 'rs62036939', 'rs7198421', 'rs73192930', 'rs73192932', 'rs73192934', 'rs73478102', 'rs73478106', 'rs73478113', 'rs73478121', 'rs73478127', 'rs73478152', 'rs73478154', 'rs73478167', 'rs73478176', 'rs73478181', 'rs74711328', 'rs7645688', 'rs79079523', 'rs9537701', 'rs9893368', 'breast_atrophy_24m'] # when I manually remove some clinc features it gave me worse result

**3.3.5 AutoEncoder feature selection (AE)**

---


Autoencoder is a type of neural network used to compress data (encode) and then reconstruct it (decode). It's unsupervised — you don’t need labels to train it.The trick is — the middle layer has fewer neurons (features), so it must learn the most important information to recreate the original. For feature selection method, it is used to extract features with Hidden Patterns. Let’s say you're working with:

Medical data with 200+ features (demographics, genomics, imaging, etc.)

Training an autoencoder to reduce this to 10–20 learned features helps:

Find underlying structure

Reduce noise/redundancy

Improve model generalization.

In [ ]:
def auto_encoder_feature_selection(df, col, feat):
  df = df.dropna(axis=0, how='any', inplace=False)
  X = df.loc[:, df.columns != col]
  Y = df[col]

  # Normalize features
  scaler = StandardScaler()
  X_scaled = scaler.fit_transform(X)

  # -------------------------------
  # Step 2: Autoencoder Architecture
  # -------------------------------
  input_dim = X_scaled.shape[1]
  encoding_dim = 20  # Set number of features you want to reduce to

  # Define the model
  input_layer = Input(shape=(input_dim,))
  #encoded = Dense(encoding_dim, activation='relu')(input_layer)
  #decoded = Dense(input_dim, activation='sigmoid')(encoded)

  encoded = keras.layers.Dense(encoding_dim, activation="relu")(input_layer)  # this keras give good than Dence but they are the same
  decoded = keras.layers.Dense(input_dim, activation="sigmoid")(encoded)



  autoencoder = Model(inputs=input_layer, outputs=decoded)
  autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
  autoencoder.summary()

  # ----------------------------
  # Step 3: Train the Autoencoder
  # ----------------------------
  X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42, stratify=Y)
  autoencoder.fit(X_train, X_train,
                  epochs=50,
                  batch_size=32,
                  shuffle=True,
                  validation_data=(X_test, X_test),
                  verbose=0)

  # ----------------------------
  # Step 4: Extract Encoded Features
  # ----------------------------
  encoder = Model(inputs=autoencoder.input, outputs=autoencoder.layers[1].output)
  encoded_features = encoder.predict(X_scaled)

  return encoded_features, Y

In [ ]:
encoded_features, Y= auto_encoder_feature_selection(reduced_df[AE_features2], dependent1, 20)

In [ ]:
encoded_features.shape

## Inside the pipline

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
import numpy as np

class AutoencoderFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, encoding_dim=10, epochs=50, batch_size=32, verbose=0):
        self.encoding_dim = encoding_dim
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose

    def fit(self, X, y=None):
        input_dim = X.shape[1]
        input_layer = Input(shape=(input_dim,))
        encoded = Dense(self.encoding_dim, activation='relu',
                        activity_regularizer=regularizers.l1(1e-5))(input_layer)
        decoded = Dense(input_dim, activation='linear')(encoded)

        self.autoencoder = Model(inputs=input_layer, outputs=decoded)
        self.encoder = Model(inputs=input_layer, outputs=encoded)

        self.autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
        self.autoencoder.fit(X, X,
                             epochs=self.epochs,
                             batch_size=self.batch_size,
                             shuffle=True,
                             verbose=self.verbose)
        return self

    def transform(self, X):
        return self.encoder.predict(X)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

def autoencoder_feature_pipeline(df, dependent, encoding_dim):
    df = df.dropna()
    X = df.drop(columns=[dependent]).select_dtypes(include=['number'])
    y = df[dependent]

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('autoencoder_selector', AutoencoderFeatureSelector(encoding_dim=encoding_dim, epochs=50)),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=5))
    ])

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
    auc_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')

    print(f"Mean AUC with Autoencoder-selected features: {auc_scores.mean():.2f}")
    return auc_scores.mean()


In [ ]:
for i in k:
  autoencoder_feature_pipeline(df_20, dependent1, i)

# Building the model

In [ ]:

# checking the features
# clinics only # without snips
# method1
feat_list1=[['history_of_heart_disease', 'depression', 'analgesics', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'delayed_healing', 'tumour_quadrant', 'tumour_locality', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_telangiectasia_tumour_bed_baseline', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['diabetes', 'history_of_heart_disease', 'ace_inhibitor', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_locality', 'tumour_histological_grade', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['history_of_heart_disease', 'depression', 'ace_inhibitor', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_histological_grade', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['history_of_heart_disease', 'hypertension', 'depression', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_hot_spots_107', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_telangiectasia_tumour_bed_baseline', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['history_of_heart_disease', 'depression', 'analgesics', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'delayed_healing', 'tumour_histological_grade', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_hot_spots_107', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['smoker', 'diabetes', 'depression', 'ace_inhibitor', 'analgesics', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['diabetes', 'history_of_heart_disease', 'depression', 'analgesics', 'surgery_type', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_hot_spots_107', 'radio_treated_breast', 'axillary_levels', 'radio_supraclavicular_fossa', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['diabetes', 'history_of_heart_disease', 'hypertension', 'depression', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'delayed_healing', 'tumour_histological_grade', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_telangiectasia_tumour_bed_baseline', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['diabetes', 'history_of_heart_disease', 'hypertension', 'depression', 'ace_inhibitor', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'tumour_histological_grade', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_hot_spots_107', 'axillary_levels', 'radio_supraclavicular_fossa', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline'],
['diabetes', 'history_of_heart_disease', 'hypertension', 'ace_inhibitor', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_hot_spots_107', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_telangiectasia_tumour_bed_baseline', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline']]


# with snps
feat_list_snp=[['nodes_examined', 'radio_supraclavicular_fossa', 'rs72857489', 'rs79223472', 'rs885170', 'radio_heart_mean_dose_Gy', 'rs72977387', 'rs10945395', 'rs461769', 'rs56402833', 'rs67342194', 'nodes_involved', 'rs61912468', 'rs6009631', 'radio_ipsilateral_lung_mean_Gy', 'rs2861705', 'radio_skin_max_dose_Gy', 'rs77405610', 'rs74853756', 'rs35410275'],
['nodes_examined', 'rs79223472', 'rs885170', 'rs67342194', 'rs72857489', 'rs56402833', 'rs461769', 'rs77405610', 'rs10945395', 'radio_supraclavicular_fossa', 'rs74853756', 'rs72977387', 'rs61912468', 'radio_heart_mean_dose_Gy', 'axillary_levels', 'radio_skin_max_dose_Gy', 'rs2954011', 'tumour_size_mm', 'rs6009631', 'rs35035095'],
['nodes_examined', 'rs72857489', 'rs67342194', 'rs79223472', 'rs10945395', 'rs56402833', 'rs35410275', 'radio_supraclavicular_fossa', 'rs74853756', 'radio_breast_ct_volume_cm3', 'rs461769', 'rs77405610', 'radio_heart_mean_dose_Gy', 'rs61912468', 'rs6009631', 'rs2954011', 'radio_skin_max_dose_Gy', 'rs1788750', 'rs633502', 'nodes_involved'],
['radio_supraclavicular_fossa', 'nodes_examined', 'rs79223472', 'rs10945395', 'rs72977387', 'rs67342194', 'rs885170', 'rs72857489', 'rs6009631', 'rs74853756', 'rs61912468', 'rs35410275', 'axillary_levels', 'breast_arm_lymphodema_baseline', 'rs56402833', 'rs1788750', 'radio_skin_max_dose_Gy', 'post_operative_haematoma', 'radio_heart_mean_dose_Gy', 'nodes_involved'],
['rs10945395', 'nodes_examined', 'rs67342194', 'radio_supraclavicular_fossa', 'rs79223472', 'rs77405610', 'rs461769', 'radio_heart_mean_dose_Gy', 'rs35410275', 'rs885170', 'rs72977387', 'rs72857489', 'breast_arm_lymphodema_baseline', 'rs1788750', 'rs6009631', 'rs56402833', 'rs10764743', 'rs61912468', 'rs2861705', 'nodes_involved'],
['rs885170', 'nodes_examined', 'rs79223472', 'rs72857489', 'rs10945395', 'radio_supraclavicular_fossa', 'rs6009631', 'rs56402833', 'rs74853756', 'nodes_involved', 'radio_breast_ct_volume_cm3', 'rs1788750', 'rs67342194', 'rs2954011', 'rs61912468', 'radio_skin_max_dose_Gy', 'rs461769', 'rs72977387', 'rs77405610', 'radio_heart_mean_dose_Gy'],
['nodes_examined', 'radio_supraclavicular_fossa', 'rs10945395', 'rs885170', 'rs79223472', 'rs67342194', 'rs72857489', 'rs6009631', 'rs461769', 'rs72977387', 'post_operative_haematoma', 'radio_heart_mean_dose_Gy', 'rs7195405', 'rs56402833', 'radio_skin_max_dose_Gy', 'breast_arm_lymphodema_baseline', 'rs74853756', 'rs2861705', 'radio_breast_ct_volume_cm3', 'rs1788750'],
['radio_supraclavicular_fossa', 'rs56402833', 'nodes_examined', 'rs10945395', 'rs79223472', 'rs6009631', 'rs77405610', 'rs461769', 'rs72977387', 'rs61912468', 'rs67342194', 'rs1788750', 'rs72857489', 'post_operative_haematoma', 'rs7195405', 'radio_skin_max_dose_Gy', 'rs2954011', 'rs885170', 'rs10764743', 'rs2861705'],
['radio_supraclavicular_fossa', 'rs79223472', 'rs10945395', 'rs67342194', 'rs72857489', 'rs6009631', 'nodes_examined', 'rs35410275', 'rs2954011', 'rs885170', 'rs461769', 'radio_breast_ct_volume_cm3', 'rs56402833', 'nodes_involved', 'rs2861705', 'radio_ipsilateral_lung_mean_Gy', 'rs74853756', 'rs77405610', 'post_operative_haematoma', 'rs1788750'],
['nodes_examined', 'radio_supraclavicular_fossa', 'rs10945395', 'rs72857489', 'rs79223472', 'rs67342194', 'rs72977387', 'rs885170', 'rs461769', 'rs74853756', 'rs6009631', 'rs77405610', 'radio_heart_mean_dose_Gy', 'rs61912468', 'rs56402833', 'rs35410275', 'nodes_involved', 'rs1788750', 'breast_swollen_arm_baseline', 'radio_breast_ct_volume_cm3']]

feat_snp_only= [['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170'],
['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170']]



for i in feat_snp_only:
  df_Top_selected = reduced_df[i + [dependent1]]
  cross_validated_feature_selection(df_Top_selected, dependent1, splits=10)

In [ ]:
Cuniq(reduced_df, dependent1)

In [ ]:
                                    # # # clinical Data # # #

lymphedema_M1= ['history_of_heart_disease', 'hypertension', 'depression', 'analgesics', 'nodes_involved', 'nodes_examined', 'post_operative_haematoma', 'post_operative_oedema', 'post_operative_infection', 'delayed_healing', 'chemo_neo_adjuvant', 'chemo_adjuvant', 'radio_heart_mean_dose_Gy', 'radio_hot_spots_107', 'axillary_levels', 'radio_supraclavicular_fossa', 'radio_photon_2nd', 'breast_telangiectasia_tumour_bed_baseline', 'breast_arm_lymphodema_baseline', 'breast_swollen_arm_baseline', dependent1]


In [ ]:
                                 # # # clinical Data and SNPs  # # #

lymphedema_M2_SNP= ['nodes_examined', 'rs72857489', 'rs67342194', 'rs79223472', 'rs10945395', 'rs56402833', 'rs35410275', 'radio_supraclavicular_fossa', 'rs74853756', 'radio_breast_ct_volume_cm3', 'rs461769', 'rs77405610', 'radio_heart_mean_dose_Gy', 'rs61912468', 'rs6009631', 'rs2954011', 'radio_skin_max_dose_Gy', 'rs1788750', 'rs633502', 'nodes_involved', dependent1]




In [ ]:
                           #### SNPs Data ####

lymphedema_SNP= ['rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170', dependent1]

In [ ]:
'''
for feat in lymphedema_M1:
  Percent_distribution(reduced_df, feat, dependent1)

'''

In [ ]:
lymphedema_M2_SNP_df= reduced_df[['nodes_examined', 'rs72857489', 'rs67342194', 'rs79223472', 'rs10945395', 'rs56402833', 'rs35410275', 'radio_supraclavicular_fossa', 'rs74853756', 'radio_breast_ct_volume_cm3', 'rs461769', 'rs77405610', 'radio_heart_mean_dose_Gy', 'rs61912468', 'rs6009631', 'rs2954011', 'radio_skin_max_dose_Gy', 'rs1788750', 'rs633502', 'nodes_involved', dependent1]]


In [ ]:
# this function to print the summary for all features against dependent

def summarize_by_hue(df, hue_col, feature_cols):
    summary_tables = {}
    counts_dict = {}
    percentages_dict = {}

    for col in feature_cols:
        # Cross-tabulate counts
        counts = pd.crosstab(df[col], df[hue_col])

        # Calculate percentages
        percentages = counts.div(counts.sum(axis=1), axis=0) * 100

        # Store each feature’s count and percent table
        counts_dict[col] = counts
        percentages_dict[col] = percentages.round(1)
        # Combine into one table
        combined = counts.astype(int).astype(str) + " (" + percentages.round(1).astype(str) + "%)"
        summary_tables[col] = combined

    return summary_tables, counts_dict, percentages_dict


In [ ]:
# if I want to check the summary of features with dependent
summary, counts, percentages = summarize_by_hue(reduced_df, dependent1, feat_cl20)

In [ ]:
#percentages['smoker']
#counts['smoker']
#summary['smoker']

for feature, table in summary.items():
    print(f"Summary for {feature}:")
    print(table)
    print("\n")


In [ ]:

with pd.ExcelWriter("armlymphedema_summary_tablesALL%20.xlsx") as writer:
    for feature, table in summary.items():
        table.to_excel(writer, sheet_name=feature[:31])  # Excel sheet name limit

In [ ]:
import re
# if above doest works usually with snips happend try with this
def clean_sheet_name(name):
    # Remove invalid characters and truncate to 31 characters
    name = re.sub(r'[:\\/*?\[\]]', '_', name)
    return name[:31]


In [ ]:
with pd.ExcelWriter("atrophy_summary_tableswithsnp.xlsx") as writer:
    for feature, table in summary.items():
        sheet_name = clean_sheet_name(feature)
        table.to_excel(writer, sheet_name=sheet_name)


In [ ]:
reduced_df_Mixfeat= [ 'rs77405610', 'rs67342194', 'rs2954011', 'rs1788750', 'rs10764743', 'rs7195405', 'rs6009631', 'rs2954007', 'rs57288348', 'rs35035095', 'rs72977387', 'rs56402833', 'rs633502', 'rs2861705', 'rs3748569', 'rs79223472', 'rs74853756', 'rs13132305', 'rs10945395', 'rs461769', 'rs61912468', 'rs72857489', 'rs35410275', 'rs885170', 'rs11631993', dependent1]

In [ ]:
lit_snp1= ['rs1037091', 'rs2211845', 'rs2836019', 'rs1662988', 'rs315721', 'rs849530', 'rs158689', 'rs3176861', 'rs1554606', 'rs1143634', 'rs1143643', 'rs17561', 'rs1800925', 'rs3775203', 'rs1056890', 'rs10464063', 'rs6879285', 'rs1130379', 'rs2239702', 'rs11801866', 'rs7330636', 'rs1985242', 'rs2070762', 'rs165656', 'rs770298', 'rs9534511', dependent1]



---



# **3.4) Feature engineering:**
Create new features or transform existing ones to improve the model's performance.

In [ ]:
def cleanData(df, features, dependent):
  df_feat= df[features]
  print(df_feat.shape)
  print(df[dependent].sort_values().value_counts())
  clean_data = df_feat.dropna(axis=0, how='any', inplace=False) ## Drop all rows with missing values
  print(clean_data.shape)
  df = clean_data
  print(df[dependent].sort_values().value_counts())
  #**normalization
  x_data = df.drop([dependent], axis=1)
  #X = (x_data - np.min(x_data)) / (np.max(x_data) - np.min(x_data)).values # IF Is not working put this
  X = (x_data - np.min(x_data)) / (np.max(x_data) - np.min(x_data))
  #X = df.drop(['arm_lymphodema_24'], axis=1) # without normalising
  y = df[dependent]
  return df, X, y

In [ ]:
reduced_df = reduced_df.replace(dict(breast_arm_lymphodema_60m={0: 0, 1: 1, 2:1 , 3:1}))

In [ ]:
# call the functing to Cleaning
#df, X, y= cleanData(reduced_df, lymphedema_M2_SNP, dependent1)  # lymphedema_M2_SNP (clinic+snps)
df, X, y= cleanData(reduced_df, lymphedema_M1, dependent1) # (clinic only)
#df, X, y= cleanData(reduced_df, lymphedema_SNP, dependent1) # (snps only)
#df, X, y= cleanData(reduced_df, lit_snp1, dependent1)


In [ ]:
# call the functing to Cleaning
df_combined, X_combined, y= cleanData(reduced_df, lymphedema_M2_SNP, dependent1)  # lymphedema_M2_SNP (clinic+snps)
df_clinic, X_clinic, y= cleanData(reduced_df, lymphedema_M1, dependent1) # (clinic only)
df_snps, X_snps, y= cleanData(reduced_df, lymphedema_SNP, dependent1) # (snps only)

In [ ]:
# Step 1: find intersection of all patient IDs
common_ids = df_clinic.index.intersection(df_snps.index).intersection(df_combined.index)

# Step 2: subset all X datasets to common IDs
X_clinic_aligned   = df_clinic.loc[common_ids]
X_snps_aligned     = df_snps.loc[common_ids]
X_combined_aligned = df_combined.loc[common_ids]

# Step 3: subset y to the same patients
y_aligned = y.loc[common_ids]  # if y is a Series
# or
# y_aligned = y.values[common_indices]  # if y is a numpy array


In [ ]:
print(X_clinic_aligned.shape)
print(X_snps_aligned.shape)
print(X_combined_aligned.shape)


In [ ]:
print(X_clinic.shape)
print(X_snps.shape)
print(X_combined.shape)

In [ ]:
# This is the extended one that covers all metrics AUC, PR, F1, BALANCED SCORE WITH COHEN_D AND PVALUSE USING T_TEST
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score)
from sklearn.base import clone

def evaluate_feature_sets_extended(X_clinic, X_snps, X_combined, y, model, n_splits=10, n_repeats=5):
    # Step 1: Align datasets by common index
    common_ids = X_clinic.index.intersection(X_snps.index).intersection(X_combined.index).intersection(y.index)
    X_clinic = X_clinic.loc[common_ids]
    X_snps = X_snps.loc[common_ids]
    X_combined = X_combined.loc[common_ids]
    y = y.loc[common_ids]

    # Step 2: Setup CV
    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

    # Step 3: Prepare storage for metrics
    metrics = ["auc", "auprc", "f1", "balanced_accuracy"]
    fold_metrics = {
        "clinic": {m: [] for m in metrics},
        "snps": {m: [] for m in metrics},
        "combined": {m: [] for m in metrics}
    }

    # Step 4: Run CV
    for train_idx, test_idx in cv.split(X_clinic, y):
        Xtrain_c, Xtest_c = X_clinic.iloc[train_idx], X_clinic.iloc[test_idx]
        Xtrain_s, Xtest_s = X_snps.iloc[train_idx], X_snps.iloc[test_idx]
        Xtrain_m, Xtest_m = X_combined.iloc[train_idx], X_combined.iloc[test_idx]
        ytrain, ytest = y.iloc[train_idx], y.iloc[test_idx]

        for Xtrain, Xtest, name in zip(
            [Xtrain_c, Xtrain_s, Xtrain_m],
            [Xtest_c, Xtest_s, Xtest_m],
            ["clinic", "snps", "combined"]
        ):
            clf = clone(model)
            clf.fit(Xtrain, ytrain)

            if hasattr(clf, "predict_proba"):
                y_score = clf.predict_proba(Xtest)[:,1]
            else:
                y_score = clf.decision_function(Xtest)

            y_pred = (y_score >= 0.5).astype(int)

            fold_metrics[name]["auc"].append(roc_auc_score(ytest, y_score))
            fold_metrics[name]["auprc"].append(average_precision_score(ytest, y_score))
            fold_metrics[name]["f1"].append(f1_score(ytest, y_pred))
            fold_metrics[name]["balanced_accuracy"].append(balanced_accuracy_score(ytest, y_pred))

    # Step 5: Summarize metrics
    def summarize(arr):
        arr = np.array(arr)
        mean = np.mean(arr)
        sd = np.std(arr, ddof=1)
        ci_low = mean - 1.96 * sd / np.sqrt(len(arr))
        ci_up = mean + 1.96 * sd / np.sqrt(len(arr))
        return mean, ci_low, ci_up

    def cohens_d(x, y):
        diff = np.array(x) - np.array(y)
        return np.mean(diff) / np.std(diff, ddof=1)

    # Build results table
    results = {}
    for metric in metrics:
        for feature in ["clinic", "snps", "combined"]:
            mean, ci_low, ci_up = summarize(fold_metrics[feature][metric])
            results[f"{feature}_{metric}_mean"] = float(round(mean, 3))
            # Convert CI to plain Python floats
            results[f"{feature}_{metric}_CI"] = [float(round(ci_low, 3)), float(round(ci_up, 3))]

    # Step 6: Paired statistical tests vs combined
    for metric in metrics:
        _, p_c = stats.ttest_rel(fold_metrics["combined"][metric], fold_metrics["clinic"][metric])
        _, p_s = stats.ttest_rel(fold_metrics["combined"][metric], fold_metrics["snps"][metric])
        d_c = cohens_d(fold_metrics["combined"][metric], fold_metrics["clinic"][metric])
        d_s = cohens_d(fold_metrics["combined"][metric], fold_metrics["snps"][metric])
        results[f"p_clinic_vs_combined_{metric}"] = float(round(p_c, 4))
        results[f"cohen_d_clinic_vs_combined_{metric}"] = float(round(d_c, 3))
        results[f"p_snps_vs_combined_{metric}"] = float(round(p_s, 4))
        results[f"cohen_d_snps_vs_combined_{metric}"] = float(round(d_s, 3))



    return pd.DataFrame([results]), fold_metrics



In [ ]:
# recall the extended function that cover all metrics

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=5,
        min_samples_leaf=1, bootstrap=True, random_state=5
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100, max_depth=None, min_samples_split=4,
        bootstrap=False, random_state=5
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=50, learning_rate=0.1, max_depth=3, random_state=5
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=50, random_state=5),
    "XGBoost": XGBClassifier(
        random_state=5, use_label_encoder=False, eval_metric="logloss"
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_split=10, min_samples_leaf=5,
        criterion="entropy", random_state=5, class_weight="balanced"
    ),
    "QDA": make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
    "Naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, random_state=5)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=20)),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(100,), activation="relu", solver="adam", random_state=5)
    ),
}

# 🔹 Run all models and collect result
rows = []
for name, model in models.items():
    results, fold_aucs = evaluate_feature_sets_extended(
        X_clinic, X_snps, X_combined, y, model
    )
    row = {"Model": name}
    # Convert the first row of the results DataFrame to a dictionary
    row.update(results.iloc[0].to_dict())
    rows.append(row)

# 🔹 Store in DataFrame
results_df = pd.DataFrame(rows).set_index("Model")

# Show results
print(results_df)

# Save for later use
results_df.to_csv("armlymphedema_all_metrics_results.csv", index=True)


In [ ]:
import matplotlib.pyplot as plt

def plot_metrics_and_cohens_d_separate(fold_metrics, results_df, model_name="Model"):
    metrics = ["auc", "auprc", "f1", "balanced_accuracy"]

    # --- Figure 1: Boxplots ---
    plt.figure(figsize=(10,6))
    data = []
    labels = []
    for metric in metrics:
        for feature in ["clinic", "snps", "combined"]:
            data.append(fold_metrics[feature][metric])
            labels.append(f"{feature}_{metric}")

    plt.boxplot(data, labels=labels, patch_artist=True)
    plt.title(f"Fold-level distributions ({model_name})")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Score")
    plt.tight_layout()
    plt.show()

    # --- Figure 2: Cohen's d barplot ---
    plt.figure(figsize=(8,6))
    cohen_d_cols = [c for c in results_df.columns if "cohen_d" in c]
    cohen_d_vals = results_df.iloc[0][cohen_d_cols].values
    clean_labels = [c.replace("cohen_d_", "").replace("_vs_", " vs ").replace("_", " ").title()
                    for c in cohen_d_cols]

    plt.barh(clean_labels, cohen_d_vals, color="skyblue")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Cohen's d Effect Sizes")
    plt.xlabel("Cohen's d")
    plt.tight_layout()
    plt.show()


In [ ]:
results, folds = evaluate_feature_sets_extended(X_clinic, X_snps, X_combined, y, RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1, bootstrap=True, random_state=5))

plot_metrics_and_cohens_d_separate(folds, results, model_name="Random Forest")


In [ ]:
def plot_auc_cohens_d(fold_metrics, group1="clinic", group2="combined"):
    from scipy.stats import norm

    # Extract fold-level AUCs
    auc1 = np.array(fold_metrics[group1]["auc"])
    auc2 = np.array(fold_metrics[group2]["auc"])

    # Means and pooled std
    mean1, mean2 = auc1.mean(), auc2.mean()
    pooled_std = np.sqrt(((auc1.std(ddof=1) ** 2) + (auc2.std(ddof=1) ** 2)) / 2)
    d = (mean2 - mean1) / pooled_std

    # x-axis range
    x = np.linspace(min(mean1, mean2) - 3*pooled_std,
                    max(mean1, mean2) + 3*pooled_std, 500)

    # PDFs
    y1 = norm.pdf(x, mean1, pooled_std)
    y2 = norm.pdf(x, mean2, pooled_std)

    # Plot
    plt.figure(figsize=(8,5))
    plt.plot(x, y1, label=f"{group1} mean = {mean1:.3f}", color="blue")
    plt.plot(x, y2, label=f"{group2} mean = {mean2:.3f}", color="red", linestyle="--")

    plt.title(f"Cohen's d = {d:.2f} (AUC)\nPooled Std = {pooled_std:.3f}")
    plt.xlabel("Mean AUC")
    plt.ylabel("Density")
    plt.legend(loc= 'upper right')
    plt.show()

    return d


In [ ]:
results, folds = evaluate_feature_sets_extended(X_clinic, X_snps, X_combined, y, RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1, bootstrap=True, random_state=5))

# Example: compare combined vs clinic AUC
d_value = plot_auc_cohens_d(folds, group1="clinic", group2="combined")
print("Cohen's d:", d_value)


In [ ]:
# Using Cross Validation to Get the Best Value of k in KNN model
# The function below shows us the best value for K in KNN using cross validation

def BestK(X, y, k_values=range(1, 21)):
    scores = []
    scaler = StandardScaler()
    # Keep only the rows in y that match X's indices
    y = y.loc[y.index.intersection(X.index)]
    X = X.loc[y.index]  # Also make sure X only keeps matching rows

    X_scaled = scaler.fit_transform(X)

    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        cv_scores = cross_val_score(knn, X_scaled, y, cv=10, scoring="roc_auc")
        scores.append(np.mean(cv_scores))

    best_index = np.argmax(scores)
    best_k = k_values[best_index]
    print("Best K:", best_k)
    print("Best Score:", scores[best_index])
    return best_k

BestK (X_combined, y)
BestK (X_clinic, y)
BestK (X_snps, y)

In [ ]:
from tqdm import tqdm
def repeated_auc_pr_roc_models(X, y, kfolds):
    plt.figure(figsize=(8, 8), dpi=300)

    # Define models (with pipeline for those needing scaling)
    MLmodels = [
        RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5,
                               min_samples_leaf=1, bootstrap=True, random_state=5),
        ExtraTreesClassifier(n_estimators=100, max_depth=None, min_samples_split=4,
                             bootstrap=False, random_state=5),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3,
                                   random_state=5),
        AdaBoostClassifier(n_estimators=50, random_state=5),

        XGBClassifier(random_state=5, use_label_encoder=False, eval_metric='logloss'),

        DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5,
                               criterion="entropy", random_state=5, class_weight="balanced"),
        make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),

        make_pipeline(StandardScaler(), GaussianNB()),

        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=5)),

        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=BestK(X, y))),

        make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),

        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu',
                                                      solver='adam', random_state=5))
    ]

    colors = cm.get_cmap('tab20', len(MLmodels))
    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats=5, random_state=0)
    #cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)


    results = []
    results2 = []
    for i, classifier in enumerate(tqdm(MLmodels, desc="Evaluating models")):
        y_real = []
        y_proba = []
        aucs = []
        pr_aucs = []

        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            classifier.fit(Xtrain, ytrain)
            y_score = (classifier.predict_proba(Xtest)[:, 1]
                       if hasattr(classifier, 'predict_proba')
                       else classifier.decision_function(Xtest))

            # ROC
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            aucs.append(auc(fpr_fold, tpr_fold))

            # PR
            pr_aucs.append(average_precision_score(ytest, y_score))

            y_real.append(ytest)
            y_proba.append(y_score)

        # concatenate predictions for plotting only
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Mean metrics across folds
        auc_mean = np.mean(aucs)
        std_auc = np.std(aucs, ddof=1)
        auc_pr = np.mean(pr_aucs)

        # Build curves from pooled predictions (for plotting only)
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        precision, recall, _ = precision_recall_curve(y_real, y_proba)

        # Model name (handle pipelines)
        model_name = (type(classifier.named_steps[list(classifier.named_steps)[-1]]).__name__
                      if hasattr(classifier, 'named_steps') else type(classifier).__name__)

        results.append({
            'classifier': model_name,
            'fpr': fpr,
            'tpr': tpr,
            'auc_mean': auc_mean,
            'std_auc': std_auc,
            'precision': precision,
            'recall': recall,
            'auc_pr': auc_pr
        })

    # Sort by ROC-AUC
    results = sorted(results, key=lambda x: x['auc_mean'], reverse=True)
    # Sort by PR-AUC
    results2 = sorted(results, key=lambda x: x['auc_pr'], reverse=True)

    # ROC Plot
    for i, result in enumerate(results):
        plt.plot(result['fpr'], result['tpr'], lw=2,
                 label=f'{result["classifier"]} (ROC AUC = {result["auc_mean"]:.2f} ± {result["std_auc"]:.2f})',
                 color=colors(i))
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1.5, label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'(c) ROC Curves of repeated {kfolds}-Fold CV (Combined Data)',
              fontsize=16, weight='bold', pad=15)
    plt.grid(True, linestyle='--', linewidth=0.5, color='lightgray')
    plt.legend(loc='lower right', fontsize=12, frameon=True, edgecolor='black')
    plt.tight_layout()
    plt.show()

    # prevalence = positive class proportion
    prevalence = np.mean(y)

    # PR Plot
    plt.figure(figsize=(8, 8), dpi=300)
    for i, result in enumerate(results2):
        plt.plot(result['recall'], result['precision'], lw=2,
                 label=f'{result["classifier"]} (AUC-PR = {result["auc_pr"]:.2f})',
                 color=colors(i))
    # plot PR baseline
    plt.plot([0, 1], [prevalence, prevalence], linestyle='--', color='gray',
             lw=1.5, label=f'Chance Level = {prevalence:.2f}')
    plt.xlabel('Recall', fontsize=14)
    plt.ylabel('Precision', fontsize=14)
    plt.title(f'(c) Precision-Recall Curves of repeated {kfolds}-Fold CV (Combined Data)',
              fontsize=16, weight='bold', pad=15)
    plt.grid(True, linestyle='--', linewidth=0.5, color='lightgray')
    plt.legend(loc='upper right', fontsize=12, frameon=True, edgecolor='black')
    plt.tight_layout()
    plt.show()


In [ ]:
repeated_auc_pr_roc_models(X, y, 10)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='bwr', edgecolor='k')
plt.title("PCA Projection (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Try t-SNE or UMAP: These methods can better capture complex class boundaries.


from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='bwr', edgecolor='k')
plt.title("t-SNE Projection (2D)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# ✅ Select only first 2 features for 2D plotting
X_vis = X.iloc[:, :2].values


# ✅ Split and scale
X_train, X_test, y_train, y_test = train_test_split(X_vis, y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_scaled = scaler.transform(X_vis)

models = {
    "Logistic Regression": LogisticRegression(random_state=10),
    "Decision Tree": DecisionTreeClassifier(random_state=10),
    "Random Forest": RandomForestClassifier(random_state=10),
    "SVM": SVC(kernel='linear'),
    "KNN": KNeighborsClassifier(n_neighbors=BestK(X, y)),
    "MLP Classifier": MLPClassifier(max_iter=1000),
    "GaussianNB": GaussianNB(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "XGBoost": XGBClassifier(random_state=10),
    "AdaBoost": AdaBoostClassifier(random_state=10),
    "Gradient Boosting": GradientBoostingClassifier(random_state=10),
    "Extra Trees": ExtraTreesClassifier(random_state=10)
}

fig, axes = plt.subplots(4, 3, figsize=(20, 40))
axes = axes.ravel()

# ✅ Create mesh for contour
x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))
grid = np.c_[xx.ravel(), yy.ravel()]

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train_scaled, y_train)
    Z = model.predict(grid)
    Z = Z.reshape(xx.shape)

    ax = axes[i]
    ax.contourf(xx, yy, Z, cmap=ListedColormap(['#FFAAAA', '#AAAAFF']), alpha=0.6)
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='bwr', edgecolor='k', s=20)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()


In [ ]:
# Change This directly when selectening the feature
features= np.delete(Kbest5, -1)

In [ ]:
# chech the correlation
#df.corr()

# **3.5) Model selection:**
Choose the appropriate algorithm(s) to use for your problem. Consider factors such as the nature of the data, the size of the dataset, and the interpretability of the model

## **3.5.1) Normal statistic Method**

**3.5.1.1) Building x and y with Oridnary Least Square method (statsmodels)**

In [ ]:
# with statsmodels
x = sm.add_constant(X) # adding a constant
model = sm.OLS(y, x).fit() # # Oridnary least square method
predictions = model.predict(x)
print_model = model.summary()
print(print_model)

In [ ]:
# Printing only the p value from statmodels
p_values = sm.OLS(y, x).fit().summary2().tables[1]['P>|t|'] #
print(p_values)

**3.5.1.2) With chi square contingency**

In [ ]:
# to check the data variables in case you are seek for specific variable
#df.head(5)

The chi-square test statistic is 10.5.
The p-value is 0.0012.
The degrees of freedom are 3.
This information is useful for determining whether there is a statistically significant association between the two categorical variables represented by dependent and x.

In [ ]:
def chi_contingency(df, dependent, categorical_col):
   for x in categorical_col:
      # For Imbalanced Data to see how many value of y feature in x
      chisqt = pd.crosstab(df[dependent], df[x], margins=True)
      value = np.array([chisqt.iloc[0].values,
                        chisqt.iloc[1].values])
      print(dependent, 'and', x ,chi2_contingency(value)[0:3])

In [ ]:
#chi_contingency(df, dependent1, features)

From above for surgery_axilla_type , 1.3579100832798462e-27 is the p-value, 132.13916260784183 is the statistical value and 4 is the degree of freedom. As the p-value is less than 0.05, we reject the NULL hypothesis and assume that the variables arm_lymphodema_24 and surgery_axilla_type are dependent of each other.

**3.6.1.3) T test Method**

In [ ]:
# T test result
def t_test(df, dependent, features):
   for x in features:
    print(dependent, 'and', x , stats.ttest_rel(df[dependent], df[x]))

In [ ]:
#t_test(df, dependent1, features)

**3.6.1.4) find correlation using ANOVA Test**

In [ ]:
# firsly we need to change the column name
#df = df.rename(columns={'arm_lymphodema_24_1.0': 'arm_lymphodema'})

In [ ]:
# anova test
def anova_test(df, dependent, features):
    for x in features:
        # we can do ANOVA test between two var y and X or as list of X vars against y
        model = ols(dependent + '~' +  x , data = df).fit() #Oridnary least square method
        result_anova = sm.stats.anova_lm(model, type=2) # ANOVA Test
        print(result_anova)

In [ ]:
#anova_test(df, dependent1, features)

In [ ]:
def logitstat_model(X, y):
    # Without Standardize the features
    train_x, test_x, train_y, test_y = train_test_split(X, y, test_size = 0.20,  random_state=4, stratify= y)
    # Logistic Regression using Statsmodels
    train_x['intercept'] = 1.0
    logit = sm.Logit(train_y, train_x).fit()
    summaryStat= logit.summary()
    print(summaryStat)
    ###########
    # Caculate Test Error
    test_x['intercept'] = 1.0
    pred = logit.predict(test_x)
    pred[pred > 0.5] = 1
    pred[pred <= 0.5] = 0
    test_err = 1 - (test_y == pred).mean()
    print("test error rate: " + str(test_err))
    ############
    # performing predictions on the test dataset
    yhat = logit.predict(test_x)
    prediction = list(map(round, yhat))

    # comparing original and predicted values of y
    print('Actual values', list(test_y.values))
    print('Predictions :', prediction)

    # confusion matrix
    cm = confusion_matrix(test_y, prediction)
    print ("Confusion Matrix : \n", cm)

    # accuracy score of the model
    print('Test accuracy = ', accuracy_score(test_y, prediction))

In [ ]:
logitstat_model(X, y)

In [ ]:
# Plotting correlation with each features against arm_lymph
def logreg(x1, dependent):
    X= df[x1].values
    y= df[dependent]
    # Splitting the dataset into the Training set and Test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 4)
    X_train= X_train.reshape(-1,1)
    X_test= X_test.reshape(-1,1)
    # fitting linear regression to the training set
    regressor = LogisticRegression()
    #regressor = LinearRegression()
    regressor.fit(X_train, y_train)
    # predicting the test set result
    y_pred = regressor.predict(X_test)
    # show the accuracy values of the predictive model
    train_score=regressor.score(X_train, y_train)
    test_score=regressor.score(X_test, y_test)
    print("Training score: {}".format(train_score))
    print("Test score: {}".format(test_score))

    #coef and intercept are coefficients and intercepts resp for our model
    print("Coefficient:\n", regressor.coef_)
    print("Intercept:\n", regressor.intercept_)
    # Mean squared error
    print("Mean squared error: %.2f" % mean_squared_error(y_test, y_pred))
    # Variance score (1 for the perfect prediction)
    print("Variance score: %2f" % r2_score(y_test, y_pred))
    # visualizing the training set result
    plt.scatter(X_train, y_train, color = 'red')
    plt.plot(X_train, regressor.predict(X_train), color = 'blue')
    plt.title('(Training set)')
    plt.xlabel(x1)
    plt.ylabel('arm_lymphodema_24 grade1')
    plt.show()
    # visualizing the test set result
    plt.scatter(X_test, y_test, color = 'purple')
    plt.plot(X_train, regressor.predict(X_train), color = 'blue')
    plt.title('(Test set)')
    plt.xlabel(x1 )
    plt.ylabel('arm_lymphodema_24 grade1')
    plt.show()

In [ ]:
for f in lit_snp:
  logreg(f, dependent1)

**===============================================================================================================**

#Building Machine Learning Models:

## Model training:

In [ ]:
# Split the training set into two (training and set)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify= y, random_state=12)

In [ ]:
# Standardize the features for training and test set
sc_X = StandardScaler()
X_train = sc_X.fit_transform(X_train)
X_test = sc_X.transform(X_test)

In [ ]:
# trying to do class weight
'''import numpy as np
from collections import Counter

# Step 1: Total number of samples
n_samples = len(y_train)

# Step 2: Number of classes and count per class
n_classes = len(np.unique(y_train))
class_counts = Counter(y_train)  # {class_label: count}

# Step 3: Compute class weights
class_weights = {
    cls: n_samples / (n_classes * count)
    for cls, count in class_counts.items()
}

print("Class Weights:", class_weights)
'''

**3.7.1) Logistic regression one to one model** one to one relationship

In [ ]:


# Plotting correlation with each feature against the dependent variable
def logreg(x1, dependent):
    X = df[x1].values
    y = df[dependent]

    # Splitting the dataset into the Training set and Test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

    # Reshaping the training and test data to be 2D arrays
    X_train = X_train.reshape(-1, 1)
    X_test = X_test.reshape(-1, 1)

    # Fitting logistic regression to the training set
    regressor = LogisticRegression()
    regressor.fit(X_train, y_train)

    # Predicting the test set results
    y_pred = regressor.predict(X_test)

    # Showing the accuracy values of the predictive model
    train_score = regressor.score(X_train, y_train)
    test_score = regressor.score(X_test, y_test)
    print("Training score: {}".format(train_score))
    print("Test score: {}".format(test_score))

    # Coefficients and intercept for the model
    print("Coefficient:\n", regressor.coef_)
    print("Intercept:\n", regressor.intercept_)

    # Confusion matrix and classification report
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

    # Visualizing the training set results
    plt.scatter(X_train, y_train, color='red')
    plt.plot(X_train, regressor.predict(X_train), color='blue')
    plt.title('Training set')
    plt.xlabel(x1)
    plt.ylabel(dependent)
    plt.show()

    # Visualizing the test set results
    plt.scatter(X_test, y_test, color='purple')
    plt.plot(X_train, regressor.predict(X_train), color='blue')
    plt.title('Test set')
    plt.xlabel(x1)
    plt.ylabel(dependent)
    plt.show()


In [ ]:
for f in features:
  logreg(f, dependent1)

**3.7.3) Building with other Machine learing Models such as KNN, RF, DT, ....**

In [ ]:
def calculate_rates(confusion_matrix):
    TN, FP, FN, TP = confusion_matrix.ravel()
    tpr = TP / (TP + FN)
    fpr = FP / (FP + TN)
    tnr = TN / (TN + FP)
    return tpr, fpr, tnr, FN / (FN + TP)

In [ ]:
def calculate_tp_fp_tn_fn(confusion_matrix):
    TP = confusion_matrix[1][1]
    FP = confusion_matrix[0][1]
    TN = confusion_matrix[0][0]
    FN = confusion_matrix[1][0]
    return TP, FP, TN, FN

**===============================================================================================================**

**BUILDING WITH NORMAL TRAIN AND TEST**

In [ ]:
# this is for the features to plot coffeiecients in logistic regression

def models(clf, X, y, title= 'AUC'):
        # split the data
        X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, stratify= y, random_state=3)
        # Fit the model on traing set
        y_pred = clf.fit(X_train, Y_train).predict(X_test)  # predict testing
        x_pred= clf.fit(X_train, Y_train).predict(X_train)  # predict training
        # print model name
        print(bold("% s model" % (title)))
        # show the accuracy values of the predictive model
        print(bold("Training accuracy score:"), (metrics.accuracy_score(Y_train, x_pred)))
        print(bold("Test accuracy score:"), (metrics.accuracy_score(Y_test, y_pred)))
        #print(bold("Test specificity:"), (metrics.recall_score(Y_test, y_pred)))
        # calculate cross validation score
        scores = cross_val_score(clf, X, y, scoring='accuracy', cv=10)
        print(bold("Cross validation mean score:"), (scores.mean()))

        # Plotting the coefficient
        print(bold("\nCoefficients:\n"))                 # Coefficients
        for feat, coef in zip(features, clf.coef_[0]):   # These working for LR only
            print(" {:>20}: {}".format(feat, coef))

        #Generate the confusion matrix
        cf_matrix = confusion_matrix(Y_test, y_pred)
        print(bold("\nconfusion matrix:\n"), cf_matrix)

        # To Claculate the rates
        tpr, fpr, tnr, fnr = calculate_rates(cf_matrix)
        print("True Positive Rate (TPR):", tpr)
        print("False Positive Rate (FPR):", fpr)
        print("True Negative Rate (TNR):", tnr)
        print("False Negative Rate (FNR):", fnr)

        TP, FP, TN, FN = calculate_tp_fp_tn_fn(cf_matrix)
        print("True Positives (TP):", TP)
        print("False Positives (FP):", FP)
        print("True Negatives (TN):", TN)
        print("False Negatives (FN):", FN)
        # Calculate sensitivity and specificity
        sensitivity = TP / (TP + FN) * 100
        specificity = TN / (TN + FP) * 100

        print("Sensitivity: %.2f%%" % sensitivity)
        print("Specificity: %.2f%%" % specificity)
        ax = sns.heatmap(cf_matrix/np.sum(cf_matrix), annot=True,
                    fmt='.2%', cmap='Blues')
        ax.set_xlabel('\nPredicted Values')
        ax.set_ylabel('Actual Values ');
        ## Ticket labels - List must be in alphabetical order
        ax.xaxis.set_ticklabels(['0','1'])
        ax.yaxis.set_ticklabels(['0','1'])
        plt.show() # Display the visualization of the Confusion Matrix.
        # plotting AUC
        prob = np.array(clf.predict_proba(X_test)[:, 1])
        Y_test+= 1
        fpr, sensitivity, _ = metrics.roc_curve(Y_test, prob, pos_label=2)
        print(bold("\nAUC = {}".format(metrics.auc(fpr, sensitivity))))
        plt.scatter(fpr, fpr, c='b', marker='s')
        plt.scatter(fpr, sensitivity, c='r', marker='o')
        plt.show() # Display the visualization of the AUC.

models(LogisticRegression(random_state=10),X, y, title= 'Logistics Regression')


In [ ]:
models(LogisticRegression(random_state=10),X, y, title= 'Logistics Regression')
'''models(RandomForestClassifier(n_estimators=50, random_state=10),X, y, title= 'RF')
models(DecisionTreeClassifier(random_state=10),X, y, title= 'DT')
models(KNN(n_neighbors = best_k),X, y, title= 'KNN')
models(SVC(probability= True),X, y, title= 'SVC')
models(XGBClassifier(random_state=10),X, y, title= 'XGB')
models(ExtraTreesClassifier(random_state=10),X, y, title= 'ETC')
models(MLPClassifier(), X, y, title= 'MLPC')
models(QuadraticDiscriminantAnalysis(),X, y, title= 'QDA')
models(GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3),X, y, title= 'GBC')
models(GaussianNB(),X, y, title= 'NB')  # highest tpr
'''


**=============================================================================================================**

**3.8) Model evaluation:** Evaluate the model's performance using the testing data. Common evaluation metrics include accuracy, precision, recall, F1 score, and ROC AUC. bold text

**3.8.1) Implement the ROC curve**

In [ ]:

def ROC_Curve(X_train, X_test, y_train, y_test, s):
    plt.figure(figsize=(10, 8), dpi=150)

    # Train multiple models
    MLmodels = [
        SVC(kernel='rbf', probability=True, random_state=10),
        MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=10),
        LogisticRegression(random_state=10),
        GaussianNB(),
        ExtraTreesClassifier(random_state=10),
        RandomForestClassifier(random_state=10),
        AdaBoostClassifier(n_estimators=50, random_state=10),
        XGBClassifier(random_state=10),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=10),
        DecisionTreeClassifier(random_state=10),
        KNeighborsClassifier(n_neighbors=3),
        QuadraticDiscriminantAnalysis(),
    ]

    # Store results for sorting
    results = []

    # Train and evaluate each model
    for model in MLmodels:
        model.fit(X_train, y_train)

        # Predict probabilities or decision function
        if hasattr(model, 'predict_proba'):
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_pred_proba = model.decision_function(X_test)

        # Calculate ROC curve and AUC
        fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
        auc = roc_auc_score(y_test, y_pred_proba)

        # Calculate g-mean for each threshold
        gmeans = sqrt(tpr * (1 - fpr))
        ix = argmax(gmeans)

        # Store the results for later plotting
        results.append({
            'model_name': type(model).__name__,
            'fpr': fpr,
            'tpr': tpr,
            'auc': auc,
            'best_threshold': thresholds[ix],
            'best_fpr': fpr[ix],
            'best_tpr': tpr[ix],
        })

    # Sort results by AUC (ascending)
    results = sorted(results, key=lambda x: x['auc'])

    # Plot ROC curves in sorted order
    for res in results:
        plt.plot(res['fpr'], res['tpr'], label=f"{res['model_name']} (AUC = {res['auc']:.2f})")

        # Mark the best threshold
        plt.scatter(res['best_fpr'], res['best_tpr'], marker='o', color='red', s=50)
        plt.text(res['best_fpr'], res['best_tpr'], f'{res["best_threshold"]:.2f}', fontsize=8, color='black')

    # Plot the diagonal line (random chance level)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')

    # Set plot labels and title
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'Receiver Operating Characteristic (ROC) Curve with {s}')

    # Set grid and background color
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, linestyle='-', linewidth=1, color='white')

    # Display legend with optimized placement
    plt.legend(fontsize='small', loc='lower right', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
ROC_Curve(X_train, X_test, y_train, y_test, 'normal training and test sets')

In [ ]:
# with auto encoded
ROC_Curve(encoded_Xtrain, encoded_Xtest, ytrain, ytest, 'Encoded training and test sets')

In [ ]:

# Example classifier
clf = RandomForestClassifier(random_state=42)

# Apply 5-fold cross-validation
scores = cross_val_score(clf, encoded_Xtrain, ytrain, cv=5, scoring='roc_auc')

print("Cross-validated AUC scores:", scores)
print("Mean AUC:", np.mean(scores))


In [ ]:


# Calculate class weights
n_samples = len(y_train)
n_classes = len(np.unique(y_train))
class_counts = Counter(y_train)

class_weights = {
    cls: n_samples / (n_classes * count)
    for cls, count in class_counts.items()
}

print("Computed Class Weights:", class_weights)

# Step 2: Apply to models

# Random Forest
rf_model = RandomForestClassifier(class_weight=class_weights)
rf_model.fit(X_train, y_train)

# Logistic Regression
lr_model = LogisticRegression(class_weight=class_weights)
lr_model.fit(X_train, y_train)

# Support Vector Classifier (SVC)
svc_model = SVC(class_weight=class_weights)
svc_model.fit(X_train, y_train)


**===============================================================================================================**

In [ ]:
# with auto encoda
#PrecisionRecallCurve(encoded_Xtrain, encoded_Xtest, ytrain, ytest)

**3.5) Model selection:** Choose the appropriate algorithm(s) to use for your problem. Consider factors such as the nature of the data, the size of the dataset, and the interpretability of the model

**===============================================================================================================**

**3.9) Model Validation** Cross validation with KFold and StratifiedKFold

Cross-validation. Cross-validation is the most commonly used tuning method. It entails splitting a training dataset into ten equal parts (folds). A given model is trained on only nine folds and then tested on the tenth one

**3.9.1) Showing the Classification Report for each fold in each model**

**===============================================================================================================**

**3.9.3) ROC curve with Stratified KFold Cross Validation**

In [ ]:
# This function to plot AUC StratifiedKFold Cross validation for any ML model
# with standard division
def roc_skfold_validationAny(model, X, y, kfolds):
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    classifier = model
    roc_auc_s = []
    tprs = []
    base_fpr = np.linspace(0, 1, 101)
    fold_data = []

    # Generate distinct colors for folds
    colors = plt.cm.viridis(np.linspace(0, 1, kfolds))  # almost use the same colors
    #colors = cm.get_cmap('tab20', kfolds)


    # Collect all fold data
    for i, (train_index, test_index) in enumerate(cv.split(X, y)):
        Xtrain, ytrain = X.iloc[train_index], y.iloc[train_index]
        Xtest, ytest = X.iloc[test_index], y.iloc[test_index]

        classifier.fit(Xtrain, ytrain)
        y_score = classifier.predict_proba(Xtest)
        fpr, tpr, _ = roc_curve(ytest, y_score[:, 1])
        roc_auc = auc(fpr, tpr)

        # Store for sorting and later plotting
        fold_data.append((roc_auc, fpr, tpr, i))
        roc_auc_s.append(roc_auc)

        # For average TPR computation
        interp_tpr = np.interp(base_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)

    # Sort folds by AUC descending
    fold_data.sort(reverse=True, key=lambda x: x[0])

    # Plot
    plt.figure(figsize=(7, 7), dpi=100)
    for idx, (roc_auc, fpr, tpr, i) in enumerate(fold_data):
        plt.plot(fpr, tpr, lw=1, linestyle='-', color=colors[idx], label='Fold %d (AUC = %0.2f)' % (i+1, roc_auc))
        #plt.plot(fpr, tpr, lw=1, linestyle='-', color=colors(idx/ kfolds), label='Fold %d (AUC = %0.2f)' % (i+1, roc_auc))   # using this when taking different colors

    tprs = np.array(tprs)
    mean_tprs = tprs.mean(axis=0)
    std = tprs.std(axis=0)
    auc_mean = np.mean(roc_auc_s)
    std_auc = np.std(roc_auc_s)

    tprs_upper = np.minimum(mean_tprs + std, 1)
    tprs_lower = mean_tprs - std

    plt.plot(base_fpr, mean_tprs, color='black', linestyle='--',
             label=f'Mean ROC (AUC = {auc_mean:.2f} ± {std_auc:.2f})', lw=2)
    plt.fill_between(base_fpr, tprs_lower, tprs_upper, color='grey', alpha=0.3,
                     label='± 1 std. dev.')
    plt.plot([0, 1], [0, 1], '--', color='gray', label='No skill (AUC = 0.5)')

    plt.legend(loc='lower right')
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.title(f'{kfolds}-Fold ROC Curve for {type(model).__name__}')
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(color='white', linestyle='-', linewidth=1)
    plt.show()

    #return roc_auc_s  # Optional: return AUCs per fold


In [ ]:
# Plotting ROC Curve for cross validation models
roc_skfold_validationAny(RandomForestClassifier(random_state= 10), X, y, 10)
#roc_skfold_validationAny(LogisticRegression(random_state= 10), X, y, 5)
#roc_skfold_validationAny(KNeighborsClassifier(n_neighbors = best_k), X, y, 5)
#roc_skfold_validationAny(XGBClassifier(random_state= 10), X, y, 5)
#roc_skfold_validationAny(GaussianNB(), X, y, 5)
#roc_skfold_validationAny(DecisionTreeClassifier(random_state= 10), X, y, 10, 'DT')
#roc_skfold_validationAny(ExtraTreesClassifier(random_state= 10), X, y, 10)
#roc_skfold_validationAny(SVC(kernel='linear', probability=True, random_state= 10), X, y, 10, 'SVC Linear')  # the ROC is better in this model
#roc_skfold_validationAny(SVC(kernel='rbf', probability=True, random_state= 10), X, y, 10)

---

**Second Way: ROC curve With Stratified KFold Cross Validation for all models togathers**

In [ ]:
# This does not contain the pipeline of standarise scaling
def roc_models_old(X, y, kfolds):

    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [
        RandomForestClassifier(random_state=10),
        ExtraTreesClassifier(random_state=10),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=3),
        AdaBoostClassifier(n_estimators=50, random_state=10),
        XGBClassifier(random_state=10),
        QuadraticDiscriminantAnalysis(),
        GaussianNB(),
        SVC(kernel='rbf', probability=True, random_state=10),
        KNeighborsClassifier(n_neighbors=BestK(X, y)),
        LogisticRegression(random_state=10),
        DecisionTreeClassifier(random_state=10),
        MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=10),
    ]

    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))

    # Define cross-validator
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    # Store results for sorting
    results = []

    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # Fit classifier and predict probabilities
            classifier.fit(Xtrain, ytrain)
            y_score = (classifier.predict_proba(Xtest)[:, 1]
                       if hasattr(classifier, 'predict_proba')
                       else classifier.decision_function(Xtest))

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)

        # Store results for later sorting
        results.append({'classifier': type(classifier).__name__, 'fpr': fpr, 'tpr': tpr, 'auc_mean': auc_mean, 'std_auc': std_auc})

    # Sort results by AUC mean
    results = sorted(results, key=lambda x: x['auc_mean'])

    # Plot ROC curves in sorted order
    for i, result in enumerate(results):
        fpr, tpr, auc_mean, std_auc = result['fpr'], result['tpr'], result['auc_mean'], result['std_auc']
        plt.plot(fpr, tpr, label=f'{result["classifier"]} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i))

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()

In [ ]:
roc_models_old(X, y, 10)

# Cross Validation

In [ ]:
plt.rcdefaults()               # Reset ALL matplotlib settings to default
plt.style.use('default')       # Use clean default style
def roc_models_new(X, y, kfolds):
    #
    plt.figure(figsize=(10, 8), dpi=150)

    # -----------------------------
    # Define classifiers
    # -----------------------------
    MLmodels = [
        RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,
                               bootstrap=True, random_state=5),
        ExtraTreesClassifier(n_estimators=100, max_depth=None, min_samples_split=4, bootstrap=False, random_state=5),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=5),
        AdaBoostClassifier(n_estimators=50, random_state=5),
        XGBClassifier(random_state=5, use_label_encoder=False, eval_metric='logloss'),
        DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5,
                               criterion="entropy", random_state=5, class_weight="balanced"),
        make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
        make_pipeline(StandardScaler(), GaussianNB()),
        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=5)),
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=BestK(X, y))),
        make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),
        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu',
                                                      solver='adam', random_state=5))
    ]

    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))

    # Define cross-validator
    #cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats=5, random_state=0)

    # Store results for sorting
    results = []

    # Iterate over each classifier
    for i, classifier in enumerate(tqdm(MLmodels, desc="Evaluating models")):

        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            if isinstance(X, pd.DataFrame):
                Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
                ytrain, ytest = y.iloc[train_index], y.iloc[test_index]
            else:
                Xtrain, Xtest = X[train_index], X[test_index]
                ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # Fit classifier and predict probabilities
            classifier.fit(Xtrain, ytrain)
            y_score = (classifier.predict_proba(Xtest)[:, 1]
                       if hasattr(classifier, 'predict_proba')
                       else classifier.decision_function(Xtest))

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Overall ROC and AUC (from pooled predictions)
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        overall_auc = auc(fpr, tpr)

        # Fold statistics
        mean_auc = np.mean(aucs)         # average of fold-wise AUCs
        std_auc = np.std(aucs, ddof=1)   # standard deviation of fold AUCs

        # Store results for later sorting
        results.append({
            'classifier': type(classifier.named_steps[list(classifier.named_steps)[-1]]).__name__
                          if hasattr(classifier, 'named_steps') else type(classifier).__name__,
            'fpr': fpr,
            'tpr': tpr,
            'overall_auc': overall_auc,
            'mean_auc': mean_auc,
            'std_auc': std_auc
        })

    # Sort results by mean AUC
    results = sorted(results, key=lambda x: x['mean_auc'], reverse=True)

    plt.figure(figsize=(8, 8), dpi=300)

    # Plot ROC curves
    for i, result in enumerate(results):
        fpr, tpr = result['fpr'], result['tpr']
        mean_auc, std_auc = result['mean_auc'], result['std_auc']
        plt.plot(fpr, tpr, lw=2,
                label=f'{result["classifier"]} (Mean AUC={mean_auc:.2f} ± {std_auc:.2f})',
                color=colors(i))

    # Diagonal line
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1.5, label='Chance (AUC=0.5)')

    # Labels and formatting
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'(c) ROC Curves of repeated {kfolds}-Fold CV (Combined Data)',
              fontsize=16, weight='bold', pad=15)

    # Grid and legend
    plt.grid(True, linestyle='--', linewidth=0.5, color='lightgray')
    plt.legend(loc='lower right', fontsize=10, frameon=True)

    plt.tight_layout()
    plt.show()

In [ ]:
roc_models_new(X, y, 10)  # FEATURES CAME FROM FEATURES SELECTION METHOD

In [ ]:
from scipy.stats import norm
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import RepeatedStratifiedKFold

def roc_models_side_effect_repeated(X, y, kfolds):
    # -----------------------------
    # Define classifiers
    # -----------------------------
    MLmodels = [
        RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,
                               bootstrap=True, random_state=5),
        ExtraTreesClassifier(n_estimators=100, max_depth=None, min_samples_split=4, bootstrap=False, random_state=5),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=5),
        AdaBoostClassifier(n_estimators=50, random_state=5),
        XGBClassifier(random_state=5, use_label_encoder=False, eval_metric='logloss'),
        DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5,
                               criterion="entropy", random_state=5, class_weight="balanced"),
        make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
        make_pipeline(StandardScaler(), GaussianNB()),
        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=5)),
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=BestK(X, y))),
        make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),
        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu',
                                                      solver='adam', random_state=5))
    ]

    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats=5, random_state=0)
    results = []

    for clf in tqdm(MLmodels, desc="Evaluating models"):
        fold_aucs = []
        y_real_all, y_proba_all = [], []

        for train_idx, test_idx in cv.split(X, y):
            Xtrain = X.iloc[train_idx] if isinstance(X, pd.DataFrame) else X[train_idx]
            Xtest = X.iloc[test_idx] if isinstance(X, pd.DataFrame) else X[test_idx]
            ytrain = y.iloc[train_idx]
            ytest = y.iloc[test_idx]

            clf.fit(Xtrain, ytrain)
            y_score = clf.predict_proba(Xtest)[:, 1] if hasattr(clf, 'predict_proba') else clf.decision_function(Xtest)

            # Fold AUC
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            fold_aucs.append(auc(fpr_fold, tpr_fold))

            y_real_all.append(ytest)
            y_proba_all.append(y_score)

        # Overall AUC from concatenated predictions
        y_real_concat = np.concatenate(y_real_all)
        y_proba_concat = np.concatenate(y_proba_all)
        fpr, tpr, _ = roc_curve(y_real_concat, y_proba_concat)
        overall_auc = auc(fpr, tpr)

        # Fold-wise mean AUC ± 95% CI
        fold_aucs = np.array(fold_aucs)
        mean_auc = fold_aucs.mean()
        std_auc = fold_aucs.std(ddof=1)
        se_auc = std_auc / np.sqrt(kfolds)
        z = norm.ppf(0.975)
        ci_lower = mean_auc - z * se_auc
        ci_upper = mean_auc + z * se_auc

        results.append({
            'classifier': type(clf.named_steps[list(clf.named_steps)[-1]]).__name__
                          if hasattr(clf, 'named_steps') else type(clf).__name__,
            'overall_auc': overall_auc,
            'mean_auc': mean_auc,
            'std_auc': std_auc,
            'se_auc': se_auc,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper
        })

    # Sort results
    df_results = pd.DataFrame(results).sort_values(by='mean_auc', ascending=False)
    df_results = df_results.round(3)

    # Save results to Excel (clean table, no fpr/tpr)
    df_results.to_excel("5repeated_rmLymphedema_snp.xlsx", index=False)

    return df_results


In [ ]:
roc_models_side_effect_repeated(X, y, 10)

In [ ]:
from scipy.stats import norm

def roc_models_side_effectfull(X, y, kfolds):
    plt.figure(figsize=(10, 8), dpi=150)

    # -----------------------------
    # Define classifiers
    # -----------------------------
    MLmodels = [
        RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1,
                               bootstrap=True, random_state=5),
        ExtraTreesClassifier(n_estimators=100, max_depth=None, min_samples_split=4, bootstrap=False, random_state=5),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=5),
        AdaBoostClassifier(n_estimators=50, random_state=5),
        XGBClassifier(random_state=5, use_label_encoder=False, eval_metric='logloss'),
        DecisionTreeClassifier(max_depth=5, min_samples_split=10, min_samples_leaf=5,
                               criterion="entropy", random_state=5, class_weight="balanced"),
        make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
        make_pipeline(StandardScaler(), GaussianNB()),
        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=5)),
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=BestK(X, y))),
        make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),
        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu',
                                                      solver='adam', random_state=5))
    ]

    colors = cm.get_cmap('tab20', len(MLmodels))
    #cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats=5, random_state=0)

    results = []

    for i, clf in enumerate(tqdm(MLmodels, desc="Evaluating models")):
        fold_aucs = []
        y_real_all, y_proba_all = [], []

        for train_idx, test_idx in cv.split(X, y):
            Xtrain = X.iloc[train_idx] if isinstance(X, pd.DataFrame) else X[train_idx]
            Xtest = X.iloc[test_idx] if isinstance(X, pd.DataFrame) else X[test_idx]
            ytrain = y.iloc[train_idx]
            ytest = y.iloc[test_idx]

            clf.fit(Xtrain, ytrain)

            # Prediction
            y_score = clf.predict_proba(Xtest)[:, 1] if hasattr(clf, 'predict_proba') else clf.decision_function(Xtest)

            # Fold-wise AUC
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            fold_aucs.append(auc(fpr_fold, tpr_fold))

            y_real_all.append(ytest)
            y_proba_all.append(y_score)

        # Concatenate predictions for overall ROC
        y_real_concat = np.concatenate(y_real_all)
        y_proba_concat = np.concatenate(y_proba_all)
        fpr, tpr, _ = roc_curve(y_real_concat, y_proba_concat)
        overall_auc = auc(fpr, tpr)

        # Fold-wise mean AUC ± 95% CI
        fold_aucs = np.array(fold_aucs)
        mean_auc = fold_aucs.mean()
        std_auc = fold_aucs.std(ddof=1)          # Standard Deviation
        se_auc = std_auc / np.sqrt(kfolds)       # Standard Error
        z = norm.ppf(0.975)
        ci_lower = mean_auc - z * se_auc
        ci_upper = mean_auc + z * se_auc

        results.append({
            'classifier': type(clf.named_steps[list(clf.named_steps)[-1]]).__name__
                          if hasattr(clf, 'named_steps') else type(clf).__name__,
            'fpr': fpr,
            'tpr': tpr,
            'overall_auc': overall_auc,  # concatenated predictions
            'mean_auc': mean_auc,        # fold-wise mean
            'std_auc': std_auc,          # NEW: standard deviation
            'se_auc': se_auc,            # NEW: standard error
            'ci_lower': ci_lower,
            'ci_upper': ci_upper
        })

    # Sort by fold-wise mean AUC
    results = sorted(results, key=lambda x: x['mean_auc'], reverse=True)

    # Plot ROC curves (overall AUC from concatenated predictions)
    for i, r in enumerate(results):
        plt.plot(r['fpr'], r['tpr'],
                 label=f"{r['classifier']} (overall AUC={r['overall_auc']:.2f}, "
                       f"mean AUC={r['mean_auc']:.2f}, 95% CI [{r['ci_lower']:.2f}, {r['ci_upper']:.2f}])",
                 color=colors(i))

    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curves for Side Effect Prediction ({kfolds}-Fold CV)', fontsize=12, fontweight='bold')
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')
    plt.show()
    # Convert to DataFrame (exclude fpr/tpr arrays to keep table clean)
    df_results = pd.DataFrame(results)
    df_results = df_results.round(3)  # nicer formatting

    # Save results to Excel
    df_results.to_excel("rmLymphedema_clinic.xlsx", index=False)
    # Return DataFrame with SD and SE included
    return pd.DataFrame(results).sort_values(by='mean_auc', ascending=False)


In [ ]:
roc_models_side_effectfull(X, y, 10)

In [ ]:
# WITH STANDARD ERROR
def roc_models_newSE(X, y, kfolds):
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [
        RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1, bootstrap=True, random_state=5),
        ExtraTreesClassifier(n_estimators=100,  max_depth=None,min_samples_split=4, bootstrap=False, random_state=5),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=5),
        AdaBoostClassifier(n_estimators=50, random_state=5),
        XGBClassifier(random_state=10, use_label_encoder=False, eval_metric='logloss'),
        DecisionTreeClassifier(max_depth=10, min_samples_split=4, min_samples_leaf=2, criterion='gini', random_state=5),

        make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
        make_pipeline(StandardScaler(), GaussianNB()),
        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=5)),
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=BestK(X, y))),
        make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),
        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=5))
    ]

    colors = cm.get_cmap('tab20', len(MLmodels))
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    results = []

    for i, classifier in enumerate(tqdm(MLmodels, desc="Evaluating models")):

        y_real = []
        y_proba = []
        aucs = []

        for train_index, test_index in cv.split(X, y):
            if isinstance(X, pd.DataFrame):
                Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
                ytrain, ytest = y.iloc[train_index], y.iloc[test_index]
            else:
                Xtrain, Xtest = X[train_index], X[test_index]
                ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            classifier.fit(Xtrain, ytrain)
            y_score = (
                classifier.predict_proba(Xtest)[:, 1]
                if hasattr(classifier, 'predict_proba')
                else classifier.decision_function(Xtest)
            )

            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)

            y_real.append(ytest)
            y_proba.append(y_score)

        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)

        # Compute Standard Error of the Mean (SEM) for AUC
        sem_auc = np.std(aucs, ddof=1) / np.sqrt(len(aucs))

        results.append({
            'classifier': type(classifier.named_steps[list(classifier.named_steps)[-1]]).__name__
                          if hasattr(classifier, 'named_steps') else type(classifier).__name__,
            'fpr': fpr,
            'tpr': tpr,
            'auc_mean': auc_mean,
            'sem_auc': sem_auc
        })

    # Sort results by AUC mean
    results = sorted(results, key=lambda x: x['auc_mean'])

    # Plot ROC curves
    for i, result in enumerate(results):
        fpr, tpr, auc_mean, sem_auc = result['fpr'], result['tpr'], result['auc_mean'], result['sem_auc']
        plt.plot(fpr, tpr, label=f'{result["classifier"]} (Mean AUC = {auc_mean:.2f} ± {sem_auc:.2f})', color=colors(i))

    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation with SEM)', fontsize=12, fontweight='bold')
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')
    plt.show()

    # Optionally return results
    # return pd.DataFrame(results).sort_values(by='auc_mean', ascending=False)


In [ ]:
roc_models_newSE(X, y, 10)  # FEATURES CAME FROM FEATURES SELECTION METHOD

In [ ]:
encoded_features, Y= auto_encoder_feature_selection(reduced_df[AE_features2], dependent1, 20)

In [ ]:
roc_models_new(encoded_features, Y, 10) # FEATURES SELECTED BY AUTOENCODER

In [ ]:
# auc and precision
def auc_pr_roc_models(X, y, kfolds):
    plt.figure(figsize=(10, 8), dpi=150)

    # Define models (with pipeline for those needing scaling)
    MLmodels = [
        RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1, bootstrap=True, random_state=5),
        ExtraTreesClassifier(n_estimators=100,  max_depth=None,min_samples_split=4, bootstrap=False, random_state=5),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=5),
        AdaBoostClassifier(n_estimators=50, random_state=5),
        XGBClassifier(random_state=10, use_label_encoder=False, eval_metric='logloss'),
        DecisionTreeClassifier(max_depth=10, min_samples_split=4, min_samples_leaf=2, criterion='gini', random_state=5),

        make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
        make_pipeline(StandardScaler(), GaussianNB()),
        make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=5)),
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=BestK(X, y))),
        make_pipeline(StandardScaler(), LogisticRegression(random_state=5)),
        make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=5))
    ]

    colors = cm.get_cmap('tab20', len(MLmodels))
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    results = []
    results2=[]
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            classifier.fit(Xtrain, ytrain)
            y_score = (classifier.predict_proba(Xtest)[:, 1]
                       if hasattr(classifier, 'predict_proba')
                       else classifier.decision_function(Xtest))

            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)

            y_real.append(ytest)
            y_proba.append(y_score)

        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # ROC-AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)

        # PR-AUC
        precision, recall, _ = precision_recall_curve(y_real, y_proba)
        auc_pr = average_precision_score(y_real, y_proba)

        # Model name (handle pipelines)
        model_name = (type(classifier.named_steps[list(classifier.named_steps)[-1]]).__name__
                      if hasattr(classifier, 'named_steps') else type(classifier).__name__)

        results.append({
            'classifier': model_name,
            'fpr': fpr,
            'tpr': tpr,
            'auc_mean': auc_mean,
            'std_auc': std_auc,
            'precision': precision,
            'recall': recall,
            'auc_pr': auc_pr
        })
    # Sort by AUC
    results = sorted(results, key=lambda x: x['auc_mean'])
    results2 = sorted(results, key=lambda x: x['auc_pr'])
    # ROC Plot
    for i, result in enumerate(results):
        plt.plot(result['fpr'], result['tpr'],
                 label=f'{result["classifier"]} (ROC AUC = {result["auc_mean"]:.2f} ± {result["std_auc"]:.2f})',
                 color=colors(i))
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation)', fontsize=14, fontweight='bold')
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')
    plt.show()

    # PR Plot
    plt.figure(figsize=(10, 8), dpi=150)
    for i, result in enumerate(results2):
        plt.plot(result['recall'], result['precision'],
                 label=f'{result["classifier"]} (AUC-PR = {result["auc_pr"]:.2f})',
                 color=colors(i))
    plt.xlabel('Recall -->')
    plt.ylabel('Precision -->')
    plt.title(f'Precision-Recall Curve Comparison ({kfolds}-Fold Cross-Validation)', fontsize=14, fontweight='bold')
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)
    plt.legend(loc='upper right', fontsize='small', frameon=True, edgecolor='black')
    plt.show()


In [ ]:
auc_pr_roc_models(X, y, 10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
from sklearn.ensemble import RandomForestClassifier  # replace with your model
from sklearn.datasets import make_classification
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
plt.rcdefaults()               # Reset ALL matplotlib settings to default
plt.style.use('default')       # Use clean default style

def MCC_curve(X, y, kfold):

  # Model
  #model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, min_samples_leaf=1, bootstrap=True, random_state=5)
  #model = AdaBoostClassifier(n_estimators=50, random_state=5)
  # 10-Fold Stratified CV
  model = make_pipeline(StandardScaler(), LogisticRegression(random_state=5))
  #model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=5))
  cv = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=0)

  # Store all predictions and labels
  all_probs = []
  all_labels = []

  for train_idx, test_idx in cv.split(X, y):
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = model.predict_proba(X.iloc[test_idx])[:, 1]
    all_probs.extend(probs)
    all_labels.extend(y.iloc[test_idx])

  # Convert to numpy arrays
  all_probs = np.array(all_probs)
  all_labels = np.array(all_labels)

  # Compute MCC at different thresholds
  thresholds = np.linspace(0, 1, 200)
  mcc_scores = []

  for t in thresholds:
      preds = (all_probs >= t).astype(int)
      mcc = matthews_corrcoef(all_labels, preds)
      mcc_scores.append(mcc)

  # Step 6: Find the best threshold
  best_idx = np.argmax(mcc_scores)
  best_thresh = thresholds[best_idx]
  best_mcc = mcc_scores[best_idx]

  # Step 7: Plot MCC curve
  plt.figure(figsize=(10, 8))
  plt.plot(thresholds, mcc_scores, label='MCC Curve')
  plt.axvline(best_thresh, color='red', linestyle='--', label=f'Best Threshold = {best_thresh:.2f}')
  plt.scatter(best_thresh, best_mcc, color='red')
  plt.xlabel('Threshold')
  plt.ylabel('Matthews Correlation Coefficient')
  plt.title('MCC Curve with 10-Fold Cross-Validation')
  #plt.gca().set_facecolor('#F9F9F9')
  plt.grid(True, color='gray')
  plt.legend()
  plt.show()

  # Step 8: Print best threshold and MCC value
  print(f'Best MCC: {best_mcc:.2f} at threshold = {best_thresh:.2f}')

In [ ]:
MCC_curve(X, y, 10)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, confusion_matrix, classification_report, f1_score, precision_score, recall_score

def evaluate_model_cv_with_threshold_verbose(X, y, threshold, kfolds):
    skf = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    all_true = []
    all_probs = []
    all_preds = []
    all_indices = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        #model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5,min_samples_leaf=1, bootstrap=True, random_state=5)
        #model = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,),activation='relu', solver='adam', random_state=5))
        model = make_pipeline(StandardScaler(), LogisticRegression(random_state=5))

        model.fit(X_train, y_train)

        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs >= threshold).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)
        all_true.extend(y_test)
        all_indices.extend(X_test.index)

    # Store predictions
    results_df = X.loc[all_indices].copy()
    results_df['true_label'] = np.array(all_true)
    results_df['predicted_proba'] = np.array(all_probs)
    results_df['predicted_label'] = np.array(all_preds)

    # Metrics
    y_true = results_df['true_label']
    y_pred = results_df['predicted_label']
    mcc = matthews_corrcoef(y_true, y_pred)
    conf_matrix = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred)

    print("🔢 Confusion Matrix:\n", conf_matrix)
    print("\n📋 Classification Report:\n", report)
    print(f"✅ MCC Score at threshold {threshold}: {mcc:.4f}")

    # Friendly Summary
    tn, fp, fn, tp = conf_matrix.ravel()
    precision_0 = tn / (tn + fn) if (tn + fn) else 0
    recall_0 = tn / (tn + fp) if (tn + fp) else 0
    precision_1 = tp / (tp + fp) if (tp + fp) else 0
    recall_1 = tp / (tp + fn) if (tp + fn) else 0
    f1_1 = f1_score(y_true, y_pred)

    print("\nSummary Interpretation:")
    print("\nClass 0 – Healthy Patients")
    print(f"Precision: {precision_0:.2f} → Model is {int(precision_0 * 100)}% right when predicting 'healthy'")
    print(f"Recall:    {recall_0:.2f} → Correctly detects {int(recall_0 * 100)}% of healthy patients")

    print("\nClass 1 – Sick Patients")
    print(f"Precision: {precision_1:.2f} → Model is right {int(precision_1 * 100)}% when predicting 'sick'")
    print(f"Recall:    {recall_1:.2f} → Identifies {int(recall_1 * 100)}% of actual sick patients")
    print(f"F1 Score:   {f1_1:.2f} → Balance between precision & recall")

    print("\nOverall Performance")
    print(f"MCC Score:  {mcc:.3f} → {' Strong' if mcc > 0.6 else ' Fair' if mcc > 0.3 else ' Weak'} correlation between true and predicted")

    print("\n Sample predictions:")
    print(results_df[['true_label', 'predicted_proba', 'predicted_label']].head())

    return results_df


In [ ]:
evaluate_model_cv_with_threshold_verbose(X, y, 0.47, 10)

In [ ]:
evaluate_model_cv_with_threshold(X, y, 0.29, 10)

In [ ]:
# Finding the best parameter for all models
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, average_precision_score


def grid_search_model(X, y):
    # Create a pipeline (optional, but good if you need scaling)
    pipeline = Pipeline([
        ('scaler', StandardScaler()),          # Optional for RandomForest
        ('clf', RandomForestClassifier(random_state=42))
    ])

    # Define the hyperparameter grid
    param_grid = {
        'clf__n_estimators': [100, 200, 300],
        'clf__max_depth': [None, 10, 20],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4],
        'clf__bootstrap': [True, False]
    }

    # Use AUC-PR as the scoring metric
    scorer = make_scorer(average_precision_score, needs_proba=True)

    # Grid search
    grid_search = GridSearchCV(pipeline, param_grid, cv=10, scoring=scorer, n_jobs=-1, verbose=2)

    # Fit to your data
    grid_search.fit(X, y)

    # Best parameters and score
    print("Best Parameters:", grid_search.best_params_)
    print("Best AUC-PR Score:", grid_search.best_score_)

grid_search_model(X, y)

In [ ]:
# Finding the best parameter for all models
from sklearn.model_selection import GridSearchCV

def grid_search_models(X, y):
    models_and_grids = {
        'RandomForest': (RandomForestClassifier(random_state=12),
            {
                'n_estimators': [100, 200],
                'max_depth': [None, 10, 20],
                'min_samples_split': [2, 5],
                'min_samples_leaf': [1, 3],
            }),
        'ExtraTrees': (ExtraTreesClassifier(random_state=10),
            {
                'n_estimators': [100, 200],
                'max_depth': [None, 10],
                'min_samples_split': [2, 5],
                'min_samples_leaf': [1, 3],
            }),
        'GradientBoosting': (GradientBoostingClassifier(random_state=3),
            {
                'n_estimators': [100, 200],
                'learning_rate': [0.05, 0.1],
                'max_depth': [3, 5],
            }),
        'AdaBoost': (AdaBoostClassifier(random_state=10),
            {
                'n_estimators': [50, 100],
                'learning_rate': [0.5, 1.0],
            }),
        'XGBoost': (XGBClassifier(use_label_encoder=False,
                                  eval_metric='logloss', random_state=10),
            {
                'n_estimators': [100, 200],
                'max_depth': [3, 5],
                'learning_rate': [0.05, 0.1],
            }),
        'LogisticRegression': (Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=1000,random_state=10))]),
            {
                'clf__C': [0.1, 1, 10],
                'clf__penalty': ['l2'],
                'clf__solver': ['lbfgs']
            }),
        'SVC': (Pipeline([
                ('scaler', StandardScaler()),
                ('clf', SVC(probability=True, random_state=10))]),
            {
                'clf__C': [0.1, 1, 10],
                'clf__gamma': ['scale', 'auto'],
                'clf__kernel': ['rbf']
            }),
        'KNN': (Pipeline([
                ('scaler', StandardScaler()),
                ('clf', KNeighborsClassifier())]),
            {
                'clf__n_neighbors': [3, 5, 6, 7]
            }),
        'MLP': (Pipeline([
                ('scaler', StandardScaler()),
                ('clf', MLPClassifier(max_iter=500, random_state=10))]),
            {
                'clf__hidden_layer_sizes': [(50,), (100,), (100, 50)],
                'clf__activation': ['relu', 'tanh'],
                'clf__solver': ['adam', 'lbfgs'],
                'clf__alpha': [0.0001, 0.001],
            }),
        'DecisionTree': (DecisionTreeClassifier(random_state=10),
            {
                'max_depth': [None, 10, 20],
                'min_samples_split': [2, 5],
                'min_samples_leaf': [1, 3],
            }),
        'GaussianNB': (GaussianNB(),
            {}), # No parameters to tune
        'QDA': (QuadraticDiscriminantAnalysis(),
            {}), # No parameters to tune
    }

    print("\n Starting Grid Search...\n")

    for name, (model, param_grid) in models_and_grids.items():
        print(f" Tuning {name}...")
        grid = GridSearchCV(model, param_grid, cv=10, scoring='roc_auc', n_jobs=-1)
        grid.fit(X, y)

        print(f" Best {name} Parameters: {grid.best_params_}")
        print(f" Best {name} AUC Score: {grid.best_score_:.4f}\n")

grid_search_models(X, y)

In [ ]:
def roc_models(X, y, kfolds, best_k):
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [
    LogisticRegression(
    penalty='l2',          # Regularization type
    C=1.0,                 # Inverse of regularization strength (lower values = stronger regularization)
    solver='lbfgs',        # Efficient for small/medium datasets
    max_iter=200,          # Increased iteration count
    random_state=12
    ),
    GaussianNB(
        var_smoothing=1e-9     # Stability parameter for numerical stability
    ),
    RandomForestClassifier(
        n_estimators=100,               # Number of trees
        max_depth=10,                   # Limit depth to prevent overfitting
        min_samples_split=5,            # Minimum samples to split an internal node
        min_samples_leaf=3,             # Minimum samples per leaf node
        bootstrap=True,                 # Bootstrap sampling
        random_state=12
    ),
    MLPClassifier(
        hidden_layer_sizes=(128, 64),  # Two hidden layers (128 and 64 neurons)
        activation='relu',              # Activation function for hidden layers
        solver='adam',                  # Optimizer
        alpha=0.001,                    # L2 regularization term
        learning_rate_init=0.001,       # Initial learning rate
        max_iter=300,                   # Max iterations to ensure convergence
        random_state=12
    ),
    GradientBoostingClassifier(
        n_estimators=100,               # Number of boosting stages
        learning_rate=0.05,             # Smaller learning rate for better performance
        max_depth=4,                    # Depth of each tree
        subsample=0.8,                  # Subsample fraction for training each base learner
        min_samples_split=5,            # Minimum samples to split an internal node
        random_state=12
    ),

    QuadraticDiscriminantAnalysis(
        reg_param=0.01                  # Regularization parameter to avoid overfitting
    ),
    ExtraTreesClassifier(
        n_estimators=100,               # Number of trees
        max_depth=None,                 # Fully grown trees
        min_samples_split=4,            # Minimum samples to split an internal node
        bootstrap=False,                # No bootstrap sampling
        random_state=12
    ),
    XGBClassifier(
        n_estimators=150,               # Number of boosting rounds
        learning_rate=0.05,             # Learning rate
        max_depth=6,                    # Tree depth
        subsample=0.8,                  # Fraction of samples used per tree
        colsample_bytree=0.8,           # Subsample ratio of columns per tree
        eval_metric='logloss',          # Evaluation metric
        random_state=12,
        use_label_encoder=False         # Avoids unnecessary warnings
    ),
    AdaBoostClassifier(
        n_estimators=100,               # Number of weak classifiers
        learning_rate=0.5,              # Learning rate (higher value accelerates learning but risks overfitting)
        random_state=12
    ),
    SVC(
        kernel='rbf',                   # Non-linear kernel
        C=1.0,                          # Regularization parameter
        gamma='scale',                  # Kernel coefficient for 'rbf' and 'poly'
        probability=True,               # Enable probability estimates
        random_state=12
    ),
    KNeighborsClassifier(
        n_neighbors=best_k,
        weights='distance',             # Weight neighbors by distance
        metric='euclidean'              # Distance metric
    ),
    DecisionTreeClassifier(
        max_depth=10,                   # Limit tree depth
        min_samples_split=4,            # Minimum samples to split an internal node
        min_samples_leaf=2,             # Minimum samples per leaf node
        criterion='gini',               # Gini impurity or 'entropy'
        random_state=12
    )
]

    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))

    # Define cross-validator
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    # Store results for sorting
    results = []

    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # Fit classifier and predict probabilities
            classifier.fit(Xtrain, ytrain)
            y_score = (classifier.predict_proba(Xtest)[:, 1]
                       if hasattr(classifier, 'predict_proba')
                       else classifier.decision_function(Xtest))

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)

        # Store results for later sorting
        results.append({'classifier': type(classifier).__name__, 'fpr': fpr, 'tpr': tpr, 'auc_mean': auc_mean, 'std_auc': std_auc})

    # Sort results by AUC mean
    results = sorted(results, key=lambda x: x['auc_mean'])

    # Plot ROC curves in sorted order
    for i, result in enumerate(results):
        fpr, tpr, auc_mean, std_auc = result['fpr'], result['tpr'], result['auc_mean'], result['std_auc']
        plt.plot(fpr, tpr, label=f'{result["classifier"]} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i))

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
roc_models(X, y, 10, best_k)

**roc_bootstrap:**
This method estimates the variability of a single model's ROC curve using bootstrapping. It creates resampled datasets to calculate mean and standard deviation of the ROC curve.

* **bootstrap resampling**, repeatedly sampling the test set with replacement. This approach simulates variability in the data and evaluates the model's performance across resampled test sets
* **Errors in roc_bootstrap**
The roc_bootstrap method may raise issues such as:
Insufficient Test Set Size: If test_size in train_test_split is too small, bootstrap resampling may create overlapping or identical samples, reducing variability.
Predict Probability Assumption: Assumes that the model supports predict_proba and the output is for a binary classifier with at least two classes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

def roc_bootstrap(model, X, y, num_bootstraps, title='ROC Curve with Bootstrapping'):
    classifier = model

    # Split the data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

    # Fit the model on the training data
    classifier.fit(X_train, y_train)

    # Predict probabilities for the test set
    y_score = classifier.predict_proba(X_test)

    # Check if the model provides probabilities for binary classification
    if y_score.shape[1] >= 2:
        y_score = y_score[:, 1]  # Use the second column for binary classification
    else:
        raise ValueError("The predicted probabilities should have at least 2 columns for binary classification.")

    predicted_probabilities = y_score
    true_labels = y_test.values  # Convert y_test to a NumPy array

    # Initialize arrays for bootstrapped TPR and AUC values
    tpr_values = []
    auc_values = []
    base_fpr = np.linspace(0, 1, 101)

    for _ in range(num_bootstraps):
        # Resample with replacement
        indices = np.random.choice(len(predicted_probabilities), size=len(predicted_probabilities), replace=True)
        bootstrap_predicted_probabilities = predicted_probabilities[indices]
        bootstrap_true_labels = true_labels[indices]

        # Calculate ROC curve
        fpr, tpr, _ = roc_curve(bootstrap_true_labels, bootstrap_predicted_probabilities)

        # Interpolate TPR values to the base FPR
        tpr = np.interp(base_fpr, fpr, tpr)
        tpr[0] = 0.0  # Ensure the curve starts at 0

        tpr_values.append(tpr)
        auc_values.append(auc(base_fpr, tpr))

    # Calculate the mean and standard deviation of TPR and AUC values
    mean_tpr = np.mean(tpr_values, axis=0)
    std_tpr = np.std(tpr_values, axis=0)
    mean_auc = np.mean(auc_values)
    std_auc = np.std(auc_values)

    # Plot the mean ROC curve
    plt.figure(figsize=(10, 8), dpi=150)
    plt.plot(base_fpr, mean_tpr, color='darkorange', lw=2,
             label=f'Mean ROC Curve (AUC = {mean_auc:.2f} ± {std_auc:.2f})')

    # Add the confidence band
    plt.fill_between(base_fpr,
                     mean_tpr - std_tpr,
                     mean_tpr + std_tpr,
                     color='grey', alpha=0.2,
                     label='Confidence Band')
                     #label='Confidence Band (± 1 std)')

    # Plot the chance line
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance Level (AUC = 0.5)')

    # Set plot limits and labels
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'{title}\nModel: {type(model).__name__}, Bootstraps: {num_bootstraps}', fontsize=14, fontweight='bold')

    # Add grid and legend
    plt.grid(color='lightgrey', linestyle='--', linewidth=0.5)
    plt.legend(loc="lower right", fontsize=10, frameon=True, edgecolor='black')

    # Annotate the shaded area explanation
    #plt.text(0.6, 0.2, f'Shaded area = ± 1 std\nTPR & AUC Variability', color='grey', fontsize=10,              bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    # Show the plot
    plt.show()

# Example usage with GradientBoostingClassifier
# Assuming X and y are your features and target variables, respectively
roc_bootstrap(AdaBoostClassifier(n_estimators=100,               # Number of weak classifiers
        learning_rate=0.2,              # Learning rate (higher value accelerates learning but risks overfitting)
        random_state=12
    ), X, y, num_bootstraps=100)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights
classes = np.unique(y)
class_weights_array = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weights = {cls: weight for cls, weight in zip(classes, class_weights_array)}

print(class_weights)


In [ ]:
# trying to plot k fold cross validation for class wight but does not works
def roc_models_B(X, y, kfolds, best_k=5, class_weights):
    """
    Plots the mean AUC ROC curves for multiple machine learning models using Stratified K-Fold cross-validation.

    Parameters:
    - X: pd.DataFrame or np.ndarray
        Feature matrix.
    - y: pd.Series or np.ndarray
        Target variable.
    - kfolds: int
        Number of folds for StratifiedKFold cross-validation.
    - best_k: int, optional
        The best value of k for KNeighborsClassifier (default=5).
    - class_weights: dict or str, optional
        Weights associated with classes. Use 'balanced' for auto-calculation, or provide a dictionary like {0: w1, 1: w2}.

    Returns:
    - None. Displays the ROC plot.
    """
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [
        SVC(kernel='rbf', probability=True, random_state=10),
        GaussianNB(),
        ExtraTreesClassifier(random_state=10),
        XGBClassifier(random_state=10),
        RandomForestClassifier(random_state=10),
        AdaBoostClassifier(n_estimators=50, random_state=10),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=3),
        KNeighborsClassifier(n_neighbors=best_k),
        LogisticRegression(random_state=10),
        DecisionTreeClassifier(random_state=10),
        QuadraticDiscriminantAnalysis(),
        MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=10),
    ]

    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))

    # Define cross-validator
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    # Store results for sorting
    results = []

    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # Fit classifier and predict probabilities
            classifier.fit(Xtrain, ytrain, class_weight=class_weights)


            # Predict probabilities or decision function
            y_score = (classifier.predict_proba(Xtest)[:, 1]
                       if hasattr(classifier, 'predict_proba')
                       else classifier.decision_function(Xtest))

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)

        # Store results for later sorting
        results.append({'classifier': type(classifier).__name__, 'fpr': fpr, 'tpr': tpr, 'auc_mean': auc_mean, 'std_auc': std_auc})

    # Sort results by AUC mean
    results = sorted(results, key=lambda x: x['auc_mean'])

    # Plot ROC curves in sorted order
    for i, result in enumerate(results):
        fpr, tpr, auc_mean, std_auc = result['fpr'], result['tpr'], result['auc_mean'], result['std_auc']
        plt.plot(fpr, tpr, label=f'{result["classifier"]} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i))

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
#roc_models_B(X, y, 10, best_k, class_weights=class_weights)

In [ ]:
# This function very important for plotting cross validation to ROC curve
# This function to plot the mean average AUC StratifiedKFold Cross validation for several ML model



from matplotlib import cm

def Balanced_roc_models(X, y, kfolds):
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [

            ExtraTreesClassifier(class_weight='balanced', random_state= 10),
            HistGradientBoostingClassifier(class_weight='balanced', random_state=10),
            XGBClassifier(class_weight='balanced', random_state=10),
            RandomForestClassifier(class_weight='balanced', random_state=10),
            SVC(class_weight='balanced', kernel='rbf', probability=True, random_state= 10),
            LogisticRegression(class_weight='balanced', random_state=10),
            DecisionTreeClassifier(class_weight='balanced', random_state= 10),
            ]
    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))  # Use 'tab20' or another suitable colormap

    # Define cross-validator
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # Fit classifier and predict probabilities
            classifier.fit(Xtrain, ytrain)
            y_score = classifier.predict_proba(Xtest)[:, 1] if hasattr(classifier, 'predict_proba') else classifier.decision_function(Xtest)

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)  # Store AUC for averaging

            # plotting the folds
            #plt.plot(fpr, tpr, lw=1, linestyle='-', label='Fold %d (auc = %0.2f)' %(i+1, roc_auc)) # without dots

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)


        # Plot ROC curve for the model
        plt.plot(fpr, tpr, label=f'{type(classifier).__name__} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i)) #In the code, the std_auc is the standard deviation of the AUC values across the different folds in cross-validation. It measures the variability of the AUC (Area Under the ROC Curve) from fold to folds.

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation With Balanced classes)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
Balanced_roc_models(X, y, 10)

In [ ]:
# This function very important for plotting cross validation to ROC curve
# This function to plot the mean average AUC repeated KFold Cross validation for several ML model

def n_repeated_roc_models_balance(X, y, kfolds, n):
    plt.figure(figsize=(10, 8), dpi=150)

    # Train multiple models
    MLmodels = [
                ExtraTreesClassifier(class_weight='balanced', random_state=10),
                XGBClassifier(class_weight='balanced', random_state=10),
                HistGradientBoostingClassifier(class_weight='balanced', random_state=10),
                RandomForestClassifier(class_weight='balanced', random_state=10),
                SVC(class_weight='balanced', kernel='rbf', probability=True, random_state= 10),
                LogisticRegression(class_weight='balanced', random_state=10),
                DecisionTreeClassifier(class_weight='balanced', random_state= 10),
                ]

    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats= n, random_state=0)
    for classifier in MLmodels:
      y_real=[]
      y_proba= []
      for i, (train_index, test_index) in enumerate(cv.split(X, y)):
          Xtrain, ytrain = X.iloc[train_index], y.iloc[train_index]
          Xtest, ytest = X.iloc[test_index], y.iloc[test_index]
          classifier.fit(Xtrain, ytrain)
          y_score = classifier.predict_proba(Xtest)[:, 1] if hasattr(classifier, 'predict_proba') else classifier.decision_function(Xtest)
          fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
          roc_auc= auc(fpr_fold, tpr_fold)
          # plotting the folds
          #plt.plot(fpr, tpr, lw=1, linestyle='-', label='Fold %d (auc = %0.2f)' %(i+1, roc_auc)) # without dots
          y_real.append(ytest)
          y_proba.append(y_score)

      y_real = np.concatenate(y_real)
      y_proba= np.concatenate(y_proba)
      fpr, tpr, _ = roc_curve(y_real, y_proba)
      auc_mean= auc(fpr, tpr)
      plt.plot(fpr, tpr, label= f'{type(classifier).__name__} (auc= %0.2f)' %auc_mean)

    plt.legend(loc = 'lower right')
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    #plt.title('Repeated % d-Folds ROC Curve for several models with balanced classes' % (kfolds))
    #plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation With Balanced classes)', fontsize=14, fontweight='bold')
    plt.title(f'ROC Curve Comparison ({n}-repeats {kfolds}-Fold Cross-Validation With Balanced classes)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()

In [ ]:
n_repeated_roc_models_balance(X, y, kfolds= 10, n=3)

In [ ]:
# This is for 1 split (training set /test set) with smote Resampling method

def SMOTE_ROC_Curve(X_train, X_test, y_train, y_test):
    plt.figure(figsize=(10, 8), dpi=150)

    # # Define and Train multiple models
    MLmodels = [
        SVC(kernel='rbf', probability=True, random_state=10),
        MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=10),
        LogisticRegression(random_state=10),
        GaussianNB(),
        ExtraTreesClassifier(random_state=10),
        RandomForestClassifier(random_state=10),
        AdaBoostClassifier(n_estimators=50, random_state=10),
        XGBClassifier(random_state=10),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=10),
        DecisionTreeClassifier(random_state=10),
        KNeighborsClassifier(n_neighbors=3),
        QuadraticDiscriminantAnalysis()
    ]

    # SMOTE Method
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    # Loop over models
    for model in MLmodels:
        model.fit(X_train_res, y_train_res)
        # Predict probabilities or decision function
        if hasattr(model, 'predict_proba'):
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_pred_proba = model.decision_function(X_test)

        # Calculate ROC curve and AUC
        fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
        auc = roc_auc_score(y_test, y_pred_proba)

        # Plot ROC curve
        plt.plot(fpr, tpr, label=f'{type(model).__name__} (AUC = {auc:.2f})')

        # Calculate g-mean for each threshold
        gmeans = sqrt(tpr * (1 - fpr))
        ix = argmax(gmeans)

        # Mark the best threshold
        plt.scatter(fpr[ix], tpr[ix], marker='o', color='red', s=50)
        plt.text(fpr[ix], tpr[ix], f'{thresholds[ix]:.2f}', fontsize=8, color='black')

    # Plot the diagonal line (random chance level)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')

    # Set plot labels and title
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title('Receiver Operating Characteristic (ROC) Curve With SMOTE')

    # Set grid and background color
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, linestyle='-', linewidth=1, color='white')

    # Display legend with optimized placement
    plt.legend(fontsize='small', loc='lower right', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
SMOTE_ROC_Curve(X_train, X_test, y_train, y_test)

In [ ]:
# This function very important for plotting cross validation to ROC curve
# This function to plot the mean average AUC StratifiedKFold Cross validation for several ML model


def SMOTE_roc_models(X, y, kfolds):
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [
        RandomForestClassifier(random_state=10),
        ExtraTreesClassifier(random_state=10),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=3),
        AdaBoostClassifier(n_estimators=50, random_state=10),
        XGBClassifier(random_state=10),
        QuadraticDiscriminantAnalysis(),
        GaussianNB(),
        SVC(kernel='rbf', probability=True, random_state=10),
        KNeighborsClassifier(n_neighbors=best_k),
        LogisticRegression(random_state=10),
        DecisionTreeClassifier(random_state=10),

        MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=10),


    ]
    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))  # Use 'tab20' or another suitable colormap

    # Define cross-validator
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)

    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # # Apply SMOTE to the training data
            smote = SMOTE(random_state=42)
            X_res, y_res = smote.fit_resample(Xtrain, ytrain)

            # Fit classifier and predict probabilities
            classifier.fit(X_res, y_res)
            y_score = classifier.predict_proba(Xtest)[:, 1] if hasattr(classifier, 'predict_proba') else classifier.decision_function(Xtest)

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)  # Store AUC for averaging

            # plotting the folds
            #plt.plot(fpr, tpr, lw=1, linestyle='-', label='Fold %d (auc = %0.2f)' %(i+1, roc_auc)) # without dots

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)


        # Plot ROC curve for the model
        plt.plot(fpr, tpr, label=f'{type(classifier).__name__} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i)) #In the code, the std_auc is the standard deviation of the AUC values across the different folds in cross-validation. It measures the variability of the AUC (Area Under the ROC Curve) from fold to folds.

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation With SMOTE method)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
SMOTE_roc_models(X, y, 10)

In [ ]:
# This function very important for plotting cross validation to ROC curve
# This function to plot the mean average AUC StratifiedKFold Cross validation for several ML model


def repeated_SMOTE_roc_models(X, y, kfolds, n):
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    MLmodels = [
        ExtraTreesClassifier(random_state=10),
        GaussianNB(),
        MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', random_state=10),
        RandomForestClassifier(random_state=10),
        XGBClassifier(random_state=10),
        GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=10),
        AdaBoostClassifier(n_estimators=50, random_state=10)


    ]
    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))  # Use 'tab20' or another suitable colormap

    # Define cross-validator
    #cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats= n, random_state=0)


    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # # Apply SMOTE to the training data
            smote = SMOTE(random_state=42)
            X_res, y_res = smote.fit_resample(Xtrain, ytrain)

            # Fit classifier and predict probabilities
            classifier.fit(X_res, y_res)
            y_score = classifier.predict_proba(Xtest)[:, 1] if hasattr(classifier, 'predict_proba') else classifier.decision_function(Xtest)

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)  # Store AUC for averaging

            # plotting the folds
            #plt.plot(fpr, tpr, lw=1, linestyle='-', label='Fold %d (auc = %0.2f)' %(i+1, roc_auc)) # without dots

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)


        # Plot ROC curve for the model
        plt.plot(fpr, tpr, label=f'{type(classifier).__name__} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i)) #In the code, the std_auc is the standard deviation of the AUC values across the different folds in cross-validation. It measures the variability of the AUC (Area Under the ROC Curve) from fold to folds.

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    #plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation With SMOTE method)', fontsize=14, fontweight='bold')
    plt.title(f'ROC Curve Comparison ({n}-repeats {kfolds}-Fold Cross-Validation With SMOTE method)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
repeated_SMOTE_roc_models(X, y, kfolds= 10, n= 10)

In [ ]:
# This function very important for plotting cross validation to ROC curve
# This function to plot the mean average AUC StratifiedKFold Cross validation for several ML model

from matplotlib import cm

def repeated_SMOTE_balance_roc_models(X, y, kfolds, n):
    plt.figure(figsize=(10, 8), dpi=150)

    # List of models to evaluate
    # Train multiple models
    MLmodels = [
                ExtraTreesClassifier(class_weight='balanced', random_state=10),
                XGBClassifier(class_weight='balanced', random_state=10),
                HistGradientBoostingClassifier(class_weight='balanced', random_state=10),
                RandomForestClassifier(class_weight='balanced', random_state=10),
                SVC(class_weight='balanced', kernel='rbf', probability=True, random_state= 10),
                LogisticRegression(class_weight='balanced', random_state=10),
                DecisionTreeClassifier(class_weight='balanced', random_state= 10),
                ]
    # Create a color map with enough distinct colors for all models
    colors = cm.get_cmap('tab20', len(MLmodels))  # Use 'tab20' or another suitable colormap

    # Define cross-validator
    #cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    cv = RepeatedStratifiedKFold(n_splits=kfolds, n_repeats= n, random_state=0)


    # Iterate over each classifier
    for i, classifier in enumerate(MLmodels):
        y_real = []
        y_proba = []
        aucs = []

        # Perform cross-validation
        for train_index, test_index in cv.split(X, y):
            Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
            ytrain, ytest = y.iloc[train_index], y.iloc[test_index]

            # # Apply SMOTE to the training data
            smote = SMOTE(random_state=42)
            X_res, y_res = smote.fit_resample(Xtrain, ytrain)

            # Fit classifier and predict probabilities
            classifier.fit(X_res, y_res)
            y_score = classifier.predict_proba(Xtest)[:, 1] if hasattr(classifier, 'predict_proba') else classifier.decision_function(Xtest)

            # Calculate ROC for the current fold
            fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
            auc_fold = auc(fpr_fold, tpr_fold)
            aucs.append(auc_fold)  # Store AUC for averaging

            # plotting the folds
            #plt.plot(fpr, tpr, lw=1, linestyle='-', label='Fold %d (auc = %0.2f)' %(i+1, roc_auc)) # without dots

            # Collect true labels and probabilities for final ROC curve
            y_real.append(ytest)
            y_proba.append(y_score)

        # Concatenate results from all folds
        y_real = np.concatenate(y_real)
        y_proba = np.concatenate(y_proba)

        # Calculate overall ROC and AUC
        fpr, tpr, _ = roc_curve(y_real, y_proba)
        auc_mean = auc(fpr, tpr)
        std_auc = np.std(aucs)


        # Plot ROC curve for the model
        plt.plot(fpr, tpr, label=f'{type(classifier).__name__} (Mean AUC = {auc_mean:.2f} ± {std_auc:.2f})', color=colors(i)) #In the code, the std_auc is the standard deviation of the AUC values across the different folds in cross-validation. It measures the variability of the AUC (Area Under the ROC Curve) from fold to folds.

    # Plot formatting and labeling
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance Level = 0.5')
    plt.xlabel('False Positive Rate -->')
    plt.ylabel('True Positive Rate -->')
    #plt.title(f'ROC Curve Comparison ({kfolds}-Fold Cross-Validation With SMOTE method)', fontsize=14, fontweight='bold')
    plt.title(f'ROC Curve Comparison ({n}-repeats {kfolds}-Fold Cross-Validation With SMOTE method)', fontsize=14, fontweight='bold')

    # Background and grid settings
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(True, color='white', linestyle='-', linewidth=1)

    # Display the legend
    plt.legend(loc='lower right', fontsize='small', frameon=True, edgecolor='black')

    # Show the plot
    plt.show()


In [ ]:
repeated_SMOTE_balance_roc_models(X, y, kfolds= 10, n= 3)

**===============================================================================================================**

**3.9.4) This Function to plot precision_recall curve for each model with cross validation**

In [ ]:
# This function to plot precision recall for any ML model
def precision_recall_kf(model, X, y, kfolds, title= 'precision-recall'):

    plt.figure(figsize=(7, 7), dpi=150)
    #k_fold = KFold(n_splits=FOLDS, shuffle=True, random_state=0)
    k_fold = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    predictor = model
    y_real = []
    y_proba = []
    precision_array = []
    recall_array = np.linspace(0, 1, 100)
    for i, (train_index, test_index) in enumerate(k_fold.split(X, y)):
        Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
        ytrain, ytest = y.iloc[train_index], y.iloc[test_index]
        predictor.fit(Xtrain, ytrain)
        pred_proba = predictor.predict_proba(Xtest)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(Xtest)
        precision_fold, recall_fold, _ = precision_recall_curve(ytest, pred_proba)
        precision_array = interp(recall_array, recall_fold, precision_fold)
        pr_auc = auc(recall_fold, precision_fold)
        lab_fold = 'Fold %d AUC=%.4f' % (i+1, pr_auc)
        plt.plot(recall_fold, precision_fold, lw=1, label=lab_fold)
        y_real.append(ytest)
        y_proba.append(pred_proba)

    y_real = np.concatenate(y_real)
    y_proba = np.concatenate(y_proba)
    precision, recall, _ = precision_recall_curve(y_real, y_proba)
    pr_auc= auc(recall, precision)
    # plotting no skill line
    no_skill = len(y_proba[y_proba==1]) / len(y_proba) # this is show the no skill as a 0
    #Plot the overall average
    lab = 'Overall =%.2f' % (pr_auc)
    plt.plot(recall, precision,color='black', label=lab)
    plt.legend(loc='lower left', fontsize='small')
    #plot standard deviation # not sure if it right
    mean_precision = np.mean(precision_array)
    std_precision = np.std(precision_array)
    plt.fill_between(recall, precision + std_precision, precision - std_precision, alpha=0.3, linewidth=0, color='grey')
    # plot the roc curve for the Diagonal line
    #no_skill = len(y_real[y_real==1]) / len(y_real) # this is shows the no skill level as a 0.020
    plt.plot([0, 1], [no_skill, no_skill], linestyle='--', label='No Skill')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(loc='upper right', fontsize='small')
    plt.title('% d-Folds Precision-Recall Curve for % s model:\n' % (kfolds, title))
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(color='white', linestyle='-', linewidth=1)
    plt.show()


In [ ]:
precision_recall_kf(RandomForestClassifier(random_state= 10), X, y, 10, 'RF')
precision_recall_kf(LogisticRegression(random_state= 10), X, y, 10, 'LR')
precision_recall_kf(KNeighborsClassifier(n_neighbors = best_k), X, y, 10, 'KNN')
precision_recall_kf(XGBClassifier(random_state= 10), X, y, 10, 'XGB')
precision_recall_kf(GaussianNB(), X, y, 10, 'NB')
precision_recall_kf(DecisionTreeClassifier(random_state= 10), X, y, 10, 'DT')
precision_recall_kf(ExtraTreesClassifier(random_state= 10), X, y, 10, 'ECT')
precision_recall_kf(SVC(kernel='linear', probability=True, random_state= 10), X, y, 10, 'SVC Linear')  # the ROC is better in this model
precision_recall_kf(SVC(kernel='rbf', probability=True, random_state= 10), X, y, 10, 'SVC')

**Plotting recall precision curve for all models with K fold cross validation**

In [ ]:
# This function very important for plotting cross validation to Recall precision curve
# This function to plot the mean average AUC StratifiedKFold Cross validation for several ML model

def pr_roc_models(X, y, kfolds):
    plt.figure(figsize=(7, 7), dpi=150)
    # Train multiple models
    MLmodels = [
        LogisticRegression(random_state=10),
        RandomForestClassifier(random_state=10),
        XGBClassifier(random_state=10),
        GaussianNB(),
        DecisionTreeClassifier(random_state= 10),
        ExtraTreesClassifier(random_state= 10),
        KNeighborsClassifier(n_neighbors = best_k),
        SVC(kernel='rbf', probability=True, random_state= 10)
        ]
    auc_scores= {}
    #cv = StratifiedKFold(n_splits=kfolds)
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    for model in MLmodels:
      y_real = []
      y_proba = []
      for i, (train_index, test_index) in enumerate(cv.split(X, y)):
          Xtrain, Xtest = X.iloc[train_index], X.iloc[test_index]
          ytrain, ytest = y.iloc[train_index], y.iloc[test_index]
          model.fit(Xtrain, ytrain)
          pred_proba = model.predict_proba(Xtest)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(Xtest)
          precision_fold, recall_fold, _ = precision_recall_curve(ytest, pred_proba)
          pr_auc = auc(recall_fold, precision_fold)
          lab_fold = 'Fold %d AUC=%.4f' % (i+1, pr_auc)
          #plt.plot(recall_fold, precision_fold, lw=1, label=lab_fold)
          y_real.append(ytest)
          y_proba.append(pred_proba)

      y_real = np.concatenate(y_real)
      y_proba = np.concatenate(y_proba)
      precision, recall, _ = precision_recall_curve(y_real, y_proba)
      pr_auc= auc(recall, precision)
      # plotting no skill line
      #Plot the overall average
      lab = 'Overall =%.2f' % (pr_auc)
      #plt.plot(recall, precision, label=lab)
      plt.plot(recall, precision, label= f'{type(model).__name__} (auc= %0.3f)' %pr_auc)

    plt.legend(fontsize='small')
    plt.ylabel('Precision')
    plt.xlabel('Recall')
    plt.title('% d-Folds Precision-Recall  Curve for several models' % (kfolds))
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(color='white', linestyle='-', linewidth=1)
    plt.show()

In [ ]:
pr_roc_models(X, y, 10)

**===============================================================================================================**

**3.10) Model Improving:** Improving predictions with ensemble methods

**3.10.1) A. Voting *Classifier* score**

In [ ]:
def voting_classifier(X, y):

  # evaluate each base model
  def evaluate_models(models, X_train, X_val, y_train, y_val):
  # fit and evaluate the models
      scores = list()
      for name, model in models:
          # fit the model
          model.fit(X_train, y_train)
          # evaluate the model
          yhat = model.predict(X_val)
          acc = accuracy_score(y_val, yhat)
          # store the performance
          scores.append(acc)
      # report model performance
      return scores


  # get a list of base models
  models = list()
  models.append(('XGB', XGBClassifier(random_state=10)))
  models.append(('RF', RandomForestClassifier()))
  models.append(('nbayes', GaussianNB()))
  models.append(('GBC', GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=10)))
  models.append(('SVC', SVC(kernel='rbf', probability=True, random_state= 10)))
  models.append(('ABC', AdaBoostClassifier(n_estimators=50, random_state=10)))
  models.append(('ETC', ExtraTreesClassifier(random_state= 10)))

      # split dataset into train and test sets
  X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=4)
  # split the full train set into train and validation sets
  X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.10, random_state=4)
  # create the base models

  # fit and evaluate each model
  scores = evaluate_models(models, X_train, X_val, y_train, y_val)
  print(scores)
  # create the ensemble
  ensemble = VotingClassifier(estimators=models, voting='hard', weights=scores)
  # fit the ensemble on the training dataset
  ensemble.fit(X_train_full, y_train_full)
  # make predictions on test set
  yhat = ensemble.predict(X_test)
  # evaluate predictions
  score = accuracy_score(y_test, yhat)
  print('Weighted Avg Accuracy: %.3f' % (score*100))
  # evaluate each standalone model
  scores = evaluate_models(models, X_train_full, X_test, y_train_full, y_test)
  for i in range(len(models)):
    print('-->%s: %.3f' % (models[i][0], scores[i]*100))
  # evaluate equal weighting
  ensemble = VotingClassifier(estimators=models, voting='soft')
  ensemble.fit(X_train_full, y_train_full)
  yhat = ensemble.predict(X_test)
  score = accuracy_score(y_test, yhat)
  print('Voting Accuracy: %.3f' % (score*100))

In [ ]:
voting_classifier(X, y)

**3.10.1) B. Voting classifier AUC score**

In [ ]:
# for 1 split
def ROC_Curve_VotingClassifier(X_train, X_test, y_train, y_test):
  plt.figure(figsize=(7, 7), dpi=150)


  # Define base classifiers
  clf9 = LogisticRegression(random_state=10)
  clf1= XGBClassifier(random_state=10)
  clf2 = SVC(kernel='rbf', probability=True, random_state=10)
  clf3 = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=10)
  clf4 = RandomForestClassifier(random_state=10)
  clf5= GaussianNB()
  clf6 = AdaBoostClassifier(n_estimators=50, random_state=10)
  clf7 = KNeighborsClassifier(n_neighbors = best_k)
  clf8= ExtraTreesClassifier(random_state= 10)

  # Initialize the Voting Classifier with soft voting
  model = VotingClassifier(estimators=[('xgb', clf1), ('svm', clf2), ('gbc', clf3), ('rf', clf4),
                                       ('gb', clf5), ('abc', clf6), ('etc', clf8)], voting='soft')
  model.fit(X_train, y_train)
  y_pred_proba = model.predict_proba(X_test)[:, 1]
  auc = roc_auc_score(y_test, y_pred_proba)
  fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
  plt.plot(fpr ,tpr, label=f'{type(model).__name__} (AUC = {auc:.3f})')
  # calculate the g-mean for each threshold
  gmeans_svm = sqrt(tpr * (1-fpr))
  # locate the index of the largest g-mean
  ix = argmax(gmeans_svm)
  # plotting the optimal threshold in random forest
  plt.scatter(fpr[ix], tpr[ix], marker='*')
  plt.text(fpr[ix], tpr[ix], (thresholds[ix]).round(2), fontsize=8, weight='bold')
  plt.legend(fontsize='x-small')

  # plot the roc curve for the Diagonal line
  plt.plot([0,1], [0,1], linestyle='--',color='gray', label= 'Chance Level = 0.5')
  plt.xlabel('False Positive Rate -->')
  plt.ylabel('True Positive Rate -->')
  plt.title('Receiver Operating Characteristic (ROC) Curve')
  #plt.minorticks_on()
  plt.gca().set_facecolor('#E8E8F0')
  plt.grid(color='white', linestyle='-', linewidth=1)
  plt.show()

ROC_Curve_VotingClassifier(X_train, X_test, y_train, y_test)

**===============================================================================================================**

**3.10.2) weighted average ensemble using RUC score for training and test data**


The ensemble method used in the provided code is a form of weighted averaging ensemble. Specifically, it's an ensemble technique where the predictions of multiple base models are combined by calculating a weighted average of their predictions. The weights assigned to each model are based on their performance, typically measured using a metric like the area under the ROC curve (AUC) in this case.

In [ ]:
# for 1 split
def ROCensemble(X_train, X_test, y_train, y_test):
  plt.figure(figsize=(7, 7), dpi=100)
  # Train multiple models
  MLmodels = [
      LogisticRegression(random_state=10),
      RandomForestClassifier(random_state=10),
      XGBClassifier(random_state=10),
      GaussianNB(),
      DecisionTreeClassifier(random_state= 10),
      ExtraTreesClassifier(random_state= 10),
      KNeighborsClassifier(n_neighbors = best_k),
      SVC(kernel='rbf', probability=True, random_state= 10)
     ]

  # Dictionary to store AUC scores for each model
  auc_scores = {}

  # Train and evaluate each model
  for model in MLmodels:
      model.fit(X_train, y_train)
      y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
      auc = roc_auc_score(y_test, y_pred_proba)
      auc_scores[type(model).__name__] = auc
      fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
      plt.plot(fpr, tpr, label=f'{type(model).__name__} (AUC = {auc:.2f})')

  # Assign weights based on AUC scores
  weights = {model_name: auc / sum(auc_scores.values()) for model_name, auc in auc_scores.items()}
  print(weights)
  # Combine predictions using weighted average
  ensemble_pred_proba = np.zeros(len(y_test))
  for model in MLmodels:
      weight = weights[type(model).__name__]
      y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
      ensemble_pred_proba += y_pred_proba * weight

  # Evaluate ensemble
  ensemble_auc = roc_auc_score(y_test, ensemble_pred_proba)
  fpr, tpr, _ = roc_curve(y_test, ensemble_pred_proba)
  plt.plot(fpr, tpr, label=f'Weighted Ensemble (AUC = {ensemble_auc:.3f})', linestyle='--')

  plt.xlabel('False Positive Rate')
  plt.ylabel('True Positive Rate')
  plt.title('ROC Curve')
  plt.gca().set_facecolor('#E8E8F0')
  plt.grid(color='white', linestyle='-', linewidth=1)
  plt.legend()
  plt.show()


In [ ]:
ROCensemble(X_train, X_test, y_train, y_test)

**===============================================================================================================**

**3.10.3) weighted average ensemble the mean average ROC for all models togather with K Folds cross validation**

In [ ]:
# This function to plot AUC StratifiedKFold Cross validation for any ML model
# This function same as below but give us higher result, it a bit complex
def roc_weighted_models(X, y, kfolds):
    plt.figure(figsize=(7, 7), dpi=100)
    # Train multiple models
    MLmodels = [
        #LogisticRegression(random_state=10),
        RandomForestClassifier(random_state=10),
        XGBClassifier(random_state=10),
        GaussianNB(),
        #DecisionTreeClassifier(random_state= 10),
        ExtraTreesClassifier(random_state= 10),
        KNeighborsClassifier(n_neighbors = best_k),
        SVC(kernel='rbf', probability=True, random_state= 10)
        ]
    #auc_scores= {} same as below
    mean_auc_s= []
    m_tprs = []
    m_base_fpr = np.linspace(0, 1, 101)
    #cv = StratifiedKFold(n_splits=kfolds)
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    for classifier in MLmodels:
      roc_auc_s=[]
      tprs = []
      base_fpr = np.linspace(0, 1, 101)
      k = 1
      for i, (train_index, test_index) in enumerate(cv.split(X, y)):
          Xtrain, ytrain = X.iloc[train_index], y.iloc[train_index]
          Xtest, ytest = X.iloc[test_index], y.iloc[test_index]
          classifier.fit(Xtrain, ytrain)
          y_score = classifier.predict_proba(Xtest)
          fpr, tpr, _ = roc_curve(ytest, y_score[:, 1])
          roc_auc_s.append(auc(fpr, tpr))
          tpr = np.interp(base_fpr, fpr, tpr)
          tpr[0] = 0.0
          tprs.append(tpr)
      tprs = np.array(tprs)
      mean_tprs = tprs.mean(axis=0)
      auc_mean= np.mean(roc_auc_s)
      plt.plot(base_fpr, mean_tprs, label= f'{type(classifier).__name__} (auc= %0.3f)' %auc_mean)
      mean_auc_s.append(auc(base_fpr, mean_tprs))
      mean_tprs = np.interp(m_base_fpr, base_fpr, mean_tprs)
      mean_tprs[0] = 0.0
      m_tprs.append(mean_tprs)

    print('Mean AUC score=', mean_auc_s)
    m_tprs= np.array(m_tprs)
    average_tprs = m_tprs.mean(axis=0)
    average_auc= np.mean(mean_auc_s)

    plt.plot(m_base_fpr, average_tprs,
             color= 'black', label= 'mean ROC (auc= %0.3f)' %average_auc)

    plt.legend(loc = 'lower right')
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.title('% d-Folds ROC Curve for several models' % (kfolds))
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(color='white', linestyle='-', linewidth=1)
    plt.show()

In [ ]:
roc_weighted_models(X, y, 10)

In [ ]:
# This function to plot AUC StratifiedKFold Cross validation for any ML model

def roc_mean_weighted_models(X, y, kfolds):
    plt.figure(figsize=(7, 7), dpi=150)
    # Train multiple models
    MLmodels = [
        LogisticRegression(random_state=10),
        RandomForestClassifier(random_state=10),
        XGBClassifier(random_state=10),
        GaussianNB(),
        DecisionTreeClassifier(random_state= 10),
        ExtraTreesClassifier(random_state= 10),
        KNeighborsClassifier(n_neighbors = best_k),
        SVC(kernel='rbf', probability=True, random_state= 10)
        ]
    #auc_scores= {} same as below
    mean_auc_s= []
    m_tprs = []
    m_base_fpr = np.linspace(0, 1, 101)
    #cv = StratifiedKFold(n_splits=kfolds)
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    for classifier in MLmodels:
      roc_auc_s=[]
      y_real= []#tprs = []
      y_proba= []   #base_fp
      for i, (train_index, test_index) in enumerate(cv.split(X, y)):
          Xtrain, ytrain = X.iloc[train_index], y.iloc[train_index]
          Xtest, ytest = X.iloc[test_index], y.iloc[test_index]
          classifier.fit(Xtrain, ytrain)
          y_score = classifier.predict_proba(Xtest)[:, 1]
          fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
          roc_auc= auc(fpr_fold, tpr_fold)
          y_real.append(ytest)
          y_proba.append(y_score)

      y_real = np.concatenate(y_real)
      y_proba = np.concatenate(y_proba)
      fpr, tpr, _ = roc_curve(y_real, y_proba)
      auc_mean= auc(fpr, tpr)
      plt.plot(fpr, tpr, label= f'{type(classifier).__name__} (auc= %0.3f)' %auc_mean)
      mean_auc_s.append(auc(fpr, tpr))
      mean_tprs = np.interp(m_base_fpr, fpr, tpr)
      tpr[0] = 0.0
      m_tprs.append(mean_tprs)

    print('Mean AUC score=', mean_auc_s)
    m_tprs= np.array(m_tprs)
    average_tprs = m_tprs.mean(axis=0)
    average_auc= np.mean(mean_auc_s)

    plt.plot(m_base_fpr, average_tprs, color= 'black', label= 'mean ROC (auc= %0.3f)' %average_auc)

    plt.legend(loc = 'lower right')
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.title('% d-Folds ROC Curve for several models' % (kfolds))
    plt.gca().set_facecolor('#E8E8F0')
    plt.grid(color='white', linestyle='-', linewidth=1)
    plt.show()

In [ ]:
#roc_mean_weighted_models(X, y, 10)

**===============================================================================================================**

**3.10.4) Essemble roc AUC for K fold Cross validation**

In [ ]:
# Just Check the X_test
def essemble_roc_models(X, y, kfolds):
    plt.figure(figsize=(7, 7), dpi=150)
    # Train multiple models
    MLmodels = [
        #LogisticRegression(random_state=10),
        RandomForestClassifier(random_state=10),
        XGBClassifier(random_state=10),
        GaussianNB(),
        #DecisionTreeClassifier(random_state= 10),
        ExtraTreesClassifier(random_state= 10),
        #KNeighborsClassifier(n_neighbors = best_k),
        SVC(kernel='rbf', probability=True, random_state= 10)
        ]
    cv = StratifiedKFold(n_splits=kfolds, shuffle=True, random_state=0)
    auc_scores= {}
    #cv = StratifiedKFold(n_splits=kfolds)
    ### this (for loop) doing the cross validation for all models
    for classifier in MLmodels:
      y_real=[]
      y_proba= []
      for i, (train_index, test_index) in enumerate(cv.split(X, y)):
          Xtrain, ytrain = X.iloc[train_index], y.iloc[train_index]
          Xtest, ytest = X.iloc[test_index], y.iloc[test_index]
          classifier.fit(Xtrain, ytrain)
          y_score = classifier.predict_proba(Xtest)[:, 1]
          fpr_fold, tpr_fold, _ = roc_curve(ytest, y_score)
          roc_auc= auc(fpr_fold, tpr_fold)
          y_real.append(ytest)
          y_proba.append(y_score)

      y_real = np.concatenate(y_real)
      y_proba= np.concatenate(y_proba)
      fpr, tpr, _ = roc_curve(y_real, y_proba)
      auc_mean= auc(fpr, tpr)
      auc_scores[type(classifier).__name__] = auc_mean
      plt.plot(fpr, tpr, label= f'{type(classifier).__name__} (auc= %0.3f)' %auc_mean)

    # Assign weights based on AUC scores
    weights = {model_name: auc / sum(auc_scores.values()) for model_name, auc in auc_scores.items()}
    print(weights)
    # Combine predictions using weighted average
    ensemble_pred_proba = np.zeros(len(y_test))
    for model in MLmodels:
        weight = weights[type(model).__name__]
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test)
        ensemble_pred_proba += y_pred_proba * weight

    # Evaluate ensemble
    ensemble_auc = roc_auc_score(y_test, ensemble_pred_proba)
    fpr, tpr, _ = roc_curve(y_test, ensemble_pred_proba)
    plt.plot(fpr, tpr, label=f'Weighted Ensemble (AUC = {ensemble_auc:.2f})', linestyle='--')
    plt.legend(loc = 'lower right')
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.title('% d-Folds ROC Curve for several models' % (kfolds))
    plt.gca().set_facecolor('#E8E8F0')
    #plt.grid(True)
    plt.grid(color='white', linestyle='-', linewidth=1)
    plt.show()

In [ ]:
essemble_roc_models(X, y, 10)

**3.11) Stacking Classifier**

In [ ]:

from sklearn.ensemble import StackingClassifier



def model_stacking_classifier(X_train, X_test, y_train, y_test):
  # Define base estimators
  estimators = [
      ('lr', LogisticRegression()),
      ('dt', DecisionTreeClassifier()),
      ('svm', SVC(probability=True))
  ]
  '''
      # Define base classifiers
  clf1 = LogisticRegression(random_state=10)
  clf2 = SVC(kernel='rbf', probability=True, random_state=10)
  clf3 = DecisionTreeClassifier(random_state=10)
  clf4 = RandomForestClassifier(random_state=10)
  clf5= GaussianNB()
  clf6 = XGBClassifier(random_state=10)
  clf7 = KNeighborsClassifier(n_neighbors = best_k)
  clf8= ExtraTreesClassifier(random_state= 10)
'''
  # Initialize the Stacking Classifier
  model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
  #model = StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier())

  # Train the model
  model.fit(X_train, y_train)
  y_pred_proba = model.predict_proba(X_test)[:, 1]
  auc = roc_auc_score(y_test, y_pred_proba)
  fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
  plt.plot(fpr ,tpr, label=f'{type(model).__name__} (AUC = {auc:.3f})')
  # calculate the g-mean for each threshold
  gmeans_svm = sqrt(tpr * (1-fpr))
  # locate the index of the largest g-mean
  ix = argmax(gmeans_svm)
  # plotting the optimal threshold in random forest
  plt.scatter(fpr[ix], tpr[ix], marker='*')
  plt.text(fpr[ix], tpr[ix], (thresholds[ix]).round(2), fontsize=8, weight='bold')
  plt.legend(fontsize='x-small')

  # plot the roc curve for the Diagonal line
  plt.plot([0,1], [0,1], linestyle='--',color='gray', label= 'Chance Level = 0.5')
  plt.xlabel('False Positive Rate -->')
  plt.ylabel('True Positive Rate -->')
  plt.title('Receiver Operating Characteristic (ROC) Curve')
  #plt.minorticks_on()
  plt.gca().set_facecolor('#E8E8F0')
  plt.grid(color='white', linestyle='-', linewidth=1)
  plt.show()

model_stacking_classifier(X_train, X_test, y_train, y_test)

In [ ]:
# TRY TO DO WEGHTED AVERAGE Based on model normalise evarage above

**===============================================================================================================**

**5. Model deployment such as Batch prediction, Web service, Real-time prediction and Stream learning**

In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, confusion_matrix, classification_report
# for training and test sets
def threshold(X, y):
  # Step 1: Split your dataset
  X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

  # Step 2: Train your model
  model = RandomForestClassifier(random_state=42, class_weight='balanced')
  model.fit(X_train, y_train)

  # Step 3: Predict probabilities on test set
  probs = model.predict_proba(X_test)[:, 1]

  # Step 4: Use your optimal threshold (from MCC curve)
  optimal_threshold = 0.15  # replace with your actual best threshold
  preds = (probs >= optimal_threshold).astype(int)

  # Step 5: Evaluate the performance
  mcc = matthews_corrcoef(y_test, preds)
  conf_matrix = confusion_matrix(y_test, preds)
  report = classification_report(y_test, preds)

  print("Confusion Matrix:\n", conf_matrix)
  print("\nClassification Report:\n", report)
  print(f"MCC Score at threshold {optimal_threshold}: {mcc:.4f}")

  # Step 6: Output predictions and probabilities
  results = X_test.copy()
  results['true_label'] = y_test.values
  results['predicted_proba'] = probs
  results['predicted_label'] = preds

  # View top results
  print("\nSample predictions:")
  print(results.head())


In [ ]:
threshold(X, y)